In [1]:
# ****************************************************************************************************
# BLOCK [1]: IMPORTS, GLOBAL SILENCERS, INITIAL SETUP  
# ****************************************************************************************************

import os, warnings, torch, torch.nn as nn, torch.optim as optim
import torchaudio, torchvision
from torch.utils.data import DataLoader, Dataset
from sklearn.exceptions import UndefinedMetricWarning

# ── Silence warnings ─────────────────────────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ── New imports for adversarial attacks & explainability ──────────────────────────────────────────
import torchattacks                                        # pip install torchattacks
from captum.attr import IntegratedGradients, FeatureAblation # or SHAP if you prefer
                                                                                            
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


data_path = "/groups/hariri/nzarei/XAI_MER_Insights/IEMOCAP_full_release"

import io, contextlib, numpy as np, librosa, cv2, concurrent.futures


Using device: cuda


In [2]:
# ****************************************************************************************************
# BLOCK [2]: PREPROCESS TEXT
# ****************************************************************************************************
 # I'll define a function to collect text transcripts & labels.

def preprocess_text_iemocap(data_path):
    """
    Process textual modality from IEMOCAP dataset.
    Args:
        data_path (str): Path to the IEMOCAP dataset.
    Returns:
        text_data (list): List of text utterances.
        labels (list): Emotion labels (strings).
    """
    from transformers import BertTokenizer

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    text_data = []
    labels = []

    # Loop through sessions
    for session in range(1, 6):  # IEMOCAP typically has 5 sessions
        session_path = os.path.join(data_path, f"Session{session}", "dialog", "transcriptions")
        emo_path = os.path.join(data_path, f"Session{session}", "dialog", "EmoEvaluation")

        if not os.path.isdir(session_path):
            print(f"Warning: Could not find transcriptions in {session_path}")
            continue
        if not os.path.isdir(emo_path):
            print(f"Warning: Could not find emotion labels in {emo_path}")
            continue

        for file_name in os.listdir(session_path):
            if file_name.endswith(".txt") and not file_name.startswith("."):
                full_transcription_path = os.path.join(session_path, file_name)
                with open(full_transcription_path, "r") as f:
                    lines = f.readlines()

                # Matching transcription file to its emotion labels
                emo_file_name = file_name  # Usually the same, or adapt if naming differs
                emo_file_path = os.path.join(emo_path, emo_file_name)

                if os.path.exists(emo_file_path):
                    with open(emo_file_path, "r") as emo_f:
                        for line_number, line in enumerate(emo_f, start=1):
                            # IEMOCAP EmoEvaluation lines often have a format like:
                            # [1.1301 - 4.1908]   UXX_XX  emotion   val   act   dom
                            # We'll parse them carefully.
                            if line.startswith("[") and "]" in line:
                                parts = line.split("\t")
                                if len(parts) >= 3:
                                    emotion = parts[2].strip()
                                    # For demonstration, let's just take the first line of text
                                    # or you could match exact time stamps for each utterance.
                                    if len(lines) > 0:
                                        utterance = lines[0].strip()  # simplistic approach
                                        text_data.append(utterance)
                                        labels.append(emotion)
    return text_data, labels

print("Text preprocessing function loaded.")


Text preprocessing function loaded.


In [3]:
# ****************************************************************************************************
# BLOCK [2.5]:  LOAD ONLY THE TEXT BACKBONE  (NO audio/video yet)                
# ****************************************************************************************************
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TextBackbone(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert      = BertModel.from_pretrained(model_name)
        self.out_dim   = self.bert.config.hidden_size          # 768

    def forward(self, input_ids):
        mask = (input_ids != 0).long()
        return self.bert(input_ids, attention_mask=mask).last_hidden_state[:, 0, :]

text_backbone = TextBackbone().to(device)
print(f"✓ Text backbone ready – out_dim = {text_backbone.out_dim}")


✓ Text backbone ready – out_dim = 768


In [4]:
# ****************************************************************************************************
# BLOCK [3]:  MODEL DEFINITION  (auto-detect audio & video dims, tiny MLPs)
# ****************************************************************************************************


if False:
    import torch
    import torch.nn as nn

    # ---------- infer feature dimensions from pre-processed lists ----------
    audio_feat_dim  = len(audio_features_list[0])          # e.g. 13
    video_feat_dim  = len(video_features_list[0])          # e.g. 345 600
    TEXT_DIM        = text_backbone.out_dim                # 768
    AUDIO_OUT_DIM   = 256
    VIDEO_OUT_DIM   = 512
    FUSION_DIM      = 256
    NUM_CLASSES     = le_mm.classes_.shape[0]              # same LabelEncoder as Block 7.5

    # ---------- lightweight audio backbone ----------
    class AudioBackbone(nn.Module):
        def __init__(self, in_dim, out_dim=AUDIO_OUT_DIM):
            super().__init__()
            self.out_dim = out_dim
            self.net = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, 4 * out_dim),
                nn.ReLU(),
                nn.Linear(4 * out_dim, out_dim),
            )
        def forward(self, x):                # [B,in_dim]  or  [B,1,in_dim]
            if x.dim() == 3:
                x = x.squeeze(1)
            return self.net(x)

    # ---------- lightweight video backbone ----------
    class VideoBackbone(nn.Module):
        def __init__(self, in_dim, out_dim=VIDEO_OUT_DIM):
            super().__init__()
            self.out_dim = out_dim
            self.net = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, 2 * out_dim),
                nn.ReLU(),
                nn.Linear(2 * out_dim, out_dim),
            )
        def forward(self, x):                # [B,in_dim]  or  [B,1,in_dim]
            if x.dim() == 3:
                x = x.squeeze(1)
            return self.net(x)

    audio_backbone = AudioBackbone(audio_feat_dim).to(device)
    video_backbone = VideoBackbone(video_feat_dim).to(device)

    print(f"Audio in→out: {audio_feat_dim} → {audio_backbone.out_dim}")
    print(f"Video in→out: {video_feat_dim} → {video_backbone.out_dim}")

    # ---------- fusion + classifier ----------
    class AR_XMER_Model(nn.Module):
        def __init__(self, text_enc, audio_enc, video_enc,
                     fusion_dim=FUSION_DIM, num_classes=NUM_CLASSES):
            super().__init__()
            self.text_enc  = text_enc
            self.audio_enc = audio_enc
            self.video_enc = video_enc
            self.fusion = nn.Sequential(
                nn.Linear(text_enc.out_dim +
                          audio_enc.out_dim +
                          video_enc.out_dim, fusion_dim),
                nn.ReLU(),
                nn.Dropout(0.3),
            )
            self.classifier = nn.Linear(fusion_dim, num_classes)

        def forward(self, txt_ids, aud_vec, vid_vec):
            t = self.text_enc(txt_ids)
            a = self.audio_enc(aud_vec)
            v = self.video_enc(vid_vec)
            fused  = self.fusion(torch.cat([t, a, v], dim=1))
            logits = self.classifier(fused)
            return logits, fused

    model = AR_XMER_Model(text_backbone, audio_backbone, video_backbone).to(device)
    print("✅ Model built & moved to", device)




In [5]:
# ****************************************************************************************************
# BLOCK [4]: PREPROCESS VIDEO (Updated for prefix-based label matching)
# ****************************************************************************************************


def preprocess_video_iemocap(data_path):
    """
    Process video modality from IEMOCAP dataset.
    Match .avi video files with emotion labels using prefix matching.
    """
    video_features_list = []
    video_labels = []
    matched_count = 0
    unmatched_list = []

    for session in range(1, 6):
        # Updated path based on your actual structure
        video_dir = os.path.join(data_path, f"Session{session}", "dialog", "avi", "DivX")
        emo_dir = os.path.join(data_path, f"Session{session}", "dialog", "EmoEvaluation")

        if not os.path.isdir(video_dir):
            print(f"[VIDEO DEBUG] Video folder not found: {video_dir}")
            continue
        if not os.path.isdir(emo_dir):
            print(f"[VIDEO DEBUG] Emotion label folder not found: {emo_dir}")
            continue

        # Build clip_id -> emotion map
        label_map = {}
        for emo_file in os.listdir(emo_dir):
            if emo_file.endswith(".txt"):
                with open(os.path.join(emo_dir, emo_file), 'r') as ef:
                    for line in ef:
                        if line.startswith("[") and "]" in line:
                            parts = line.split("\t")
                            if len(parts) >= 3:
                                clip_id = parts[1].strip()
                                emotion = parts[2].strip()
                                label_map[clip_id] = emotion

        # Load and process each video
        all_videos = [f for f in os.listdir(video_dir) if f.endswith(".avi") or f.endswith(".mpg")]
        print(f"[VIDEO DEBUG] Found {len(all_videos)} total .avi/.mpg files.")

        for video_file in all_videos:
            video_path = os.path.join(video_dir, video_file)
            base_name = os.path.splitext(video_file)[0]  # e.g. Ses01F_script01_1

            # Try prefix match: clip_id.startswith(base_name)
            matched_emotion = None
            for clip_id in label_map:
                if clip_id.startswith(base_name):
                    matched_emotion = label_map[clip_id]
                    matched_count += 1
                    break

            if not matched_emotion:
                unmatched_list.append((video_file, list(label_map.keys())[:3]))  # just show first few clip_ids for debugging
                continue

            # Process video
            try:
                cap = cv2.VideoCapture(video_path)
                frame_count = 0
                frame_sum = None

                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    gray = gray.astype(np.float32) / 255.0
                    if frame_sum is None:
                        frame_sum = gray
                    else:
                        frame_sum += gray
                    frame_count += 1

                cap.release()
                if frame_count > 0 and frame_sum is not None:
                    mean_frame = frame_sum / frame_count
                    video_features = mean_frame.flatten()
                    video_features_list.append(video_features)
                    video_labels.append(matched_emotion)

            except Exception as e:
                print(f"[VIDEO DEBUG] Error processing {video_file}: {e}")

    print(f"[VIDEO DEBUG] Matched: {matched_count}, Unmatched-labeled videos: {len(unmatched_list)}")
    if unmatched_list:
        print(f"[VIDEO DEBUG] Sample unmatched video files:")
        for video_file, tried_ids in unmatched_list[:5]:
            print(f"  Unmatched: {video_file} → Tried clip_ids starting with: {tried_ids}")

    return video_features_list, video_labels

print("Video preprocessing function (with prefix matching) loaded.")


Video preprocessing function (with prefix matching) loaded.


In [6]:
# ****************************************************************************************************
# BLOCK [5]: DEFINE ADVERSARIAL ATTACKS   ← NEW BLOCK (insert after model instantiation)
# ****************************************************************************************************


if False:
    attack_fgsm = torchattacks.FGSM(model, eps=0.1)  
    attack_pgd  = torchattacks.PGD(model, eps=0.1, alpha=0.001, steps=10)

    print("FGSM & PGD attackers ready.")


In [7]:
# ****************************************************************************************************
# BLOCK [5.1]: PREPROCESS TEXT
# ****************************************************************************************************


def preprocess_text_iemocap(data_path):
    """
    Process textual modality from IEMOCAP dataset.
    Args:
        data_path (str): Path to the IEMOCAP dataset.
    Returns:
        text_data (list): List of text utterances.
        labels (list): Emotion labels (strings).
    """
    from transformers import BertTokenizer

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    text_data = []
    labels = []

    # Loop through sessions
    for session in range(1, 6):  # IEMOCAP typically has 5 sessions
        session_path = os.path.join(data_path, f"Session{session}", "dialog", "transcriptions")
        emo_path = os.path.join(data_path, f"Session{session}", "dialog", "EmoEvaluation")

        if not os.path.isdir(session_path):
            print(f"Warning: Could not find transcriptions in {session_path}")
            continue
        if not os.path.isdir(emo_path):
            print(f"Warning: Could not find emotion labels in {emo_path}")
            continue

        for file_name in os.listdir(session_path):
            if file_name.endswith(".txt") and not file_name.startswith("."):
                full_transcription_path = os.path.join(session_path, file_name)
                with open(full_transcription_path, "r") as f:
                    lines = f.readlines()

                # Matching transcription file to its emotion labels
                emo_file_name = file_name  # Usually the same, or adapt if naming differs
                emo_file_path = os.path.join(emo_path, emo_file_name)

                if os.path.exists(emo_file_path):
                    with open(emo_file_path, "r") as emo_f:
                        for line_number, line in enumerate(emo_f, start=1):
                            # IEMOCAP EmoEvaluation lines often have a format like:
                            # [1.1301 - 4.1908]   UXX_XX  emotion   val   act   dom
                            # We'll parse them carefully.
                            if line.startswith("[") and "]" in line:
                                parts = line.split("\t")
                                if len(parts) >= 3:
                                    emotion = parts[2].strip()
                                    # For demonstration, let's just take the first line of text
                                    # or you could match exact time stamps for each utterance.
                                    if len(lines) > 0:
                                        utterance = lines[0].strip()  # simplistic approach
                                        text_data.append(utterance)
                                        labels.append(emotion)
    return text_data, labels

print("Text preprocessing function loaded.")


Text preprocessing function loaded.


In [8]:
# ****************************************************************************************************
# BLOCK [5.2]: 
# ****************************************************************************************************


def preprocess_audio_iemocap_parallel(data_path, n_workers=8):
    """
    Process audio modality from IEMOCAP dataset with real label matching,
    using parallel processing to speed up feature extraction.
    
    Args:
        data_path (str): Path to the IEMOCAP dataset.
        n_workers (int): Number of parallel workers.
    Returns:
        audio_features_list (list of np.array): Extracted MFCC feature vectors.
        audio_labels (list of str): Emotion labels for each audio sample.
    """
    all_results = []
    total_wav_files = 0

    # Loop through each session
    for session in range(1, 6):
        wav_session_dir = os.path.join(data_path, f"Session{session}", "sentences", "wav")
        emo_path = os.path.join(data_path, f"Session{session}", "dialog", "EmoEvaluation")

        if not os.path.isdir(wav_session_dir):
            print(f"Warning: Wav directory not found: {wav_session_dir}")
            continue
        if not os.path.isdir(emo_path):
            print(f"Warning: Emotion directory not found: {emo_path}")
            continue

        # Build a dictionary from the EmoEvaluation files: key = wav_id, value = emotion
        audio_label_dict = {}
        for emo_file in os.listdir(emo_path):
            if emo_file.endswith(".txt"):
                emo_file_path = os.path.join(emo_path, emo_file)
                with open(emo_file_path, "r") as ef:
                    for line in ef:
                        if line.startswith("[") and "]" in line:
                            parts = line.split("\t")
                            if len(parts) >= 3:
                                wav_id = parts[1].strip()   # e.g., "Ses01F_impro01_F000"
                                emotion = parts[2].strip()  # e.g., "neutral", "angry", etc.
                                audio_label_dict[wav_id] = emotion

        # Gather list of all .wav files in this session
        wav_files = []
        for root, dirs, files in os.walk(wav_session_dir):
            for file_name in files:
                if file_name.endswith(".wav"):
                    total_wav_files += 1
                    wav_files.append(os.path.join(root, file_name))
        
        # Use parallel processing to process all .wav files in this session
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_workers) as executor:
            args_list = [(wav_path, audio_label_dict) for wav_path in wav_files]
            session_results = list(executor.map(process_wav_file, args_list))
            all_results.extend(session_results)
    
    # Filter out any failed attempts
    audio_features_list = []
    audio_labels = []
    for feat, label in all_results:
        if feat is not None:
            audio_features_list.append(feat)
            audio_labels.append(label)
    
    # Debug prints
    matched = sum(1 for lab in audio_labels if lab != "unknown")
    unknown = sum(1 for lab in audio_labels if lab == "unknown")
    print(f"\n[AUDIO PARALLEL DEBUG] Processed {total_wav_files} .wav files.")
    print(f"[AUDIO PARALLEL DEBUG] Matched: {matched}, Unknown: {unknown}")
    
    return audio_features_list, audio_labels

print("Parallel, debug-friendly audio preprocessing function loaded.")


Parallel, debug-friendly audio preprocessing function loaded.


In [9]:
# ****************************************************************************************************
# BLOCK [5.3]: PREPROCESS VIDEO (Updated for prefix-based label matching)
# ****************************************************************************************************


def preprocess_video_iemocap(data_path):
    """
    Process video modality from IEMOCAP dataset.
    Match .avi video files with emotion labels using prefix matching.
    """
    video_features_list = []
    video_labels = []
    matched_count = 0
    unmatched_list = []

    for session in range(1, 6):
        # Updated path based on your actual structure
        video_dir = os.path.join(data_path, f"Session{session}", "dialog", "avi", "DivX")
        emo_dir = os.path.join(data_path, f"Session{session}", "dialog", "EmoEvaluation")

        if not os.path.isdir(video_dir):
            print(f"[VIDEO DEBUG] Video folder not found: {video_dir}")
            continue
        if not os.path.isdir(emo_dir):
            print(f"[VIDEO DEBUG] Emotion label folder not found: {emo_dir}")
            continue

        # Build clip_id -> emotion map
        label_map = {}
        for emo_file in os.listdir(emo_dir):
            if emo_file.endswith(".txt"):
                with open(os.path.join(emo_dir, emo_file), 'r') as ef:
                    for line in ef:
                        if line.startswith("[") and "]" in line:
                            parts = line.split("\t")
                            if len(parts) >= 3:
                                clip_id = parts[1].strip()
                                emotion = parts[2].strip()
                                label_map[clip_id] = emotion

        # Load and process each video
        all_videos = [f for f in os.listdir(video_dir) if f.endswith(".avi") or f.endswith(".mpg")]
        print(f"[VIDEO DEBUG] Found {len(all_videos)} total .avi/.mpg files.")

        for video_file in all_videos:
            video_path = os.path.join(video_dir, video_file)
            base_name = os.path.splitext(video_file)[0]  # e.g. Ses01F_script01_1

            # Try prefix match: clip_id.startswith(base_name)
            matched_emotion = None
            for clip_id in label_map:
                if clip_id.startswith(base_name):
                    matched_emotion = label_map[clip_id]
                    matched_count += 1
                    break

            if not matched_emotion:
                unmatched_list.append((video_file, list(label_map.keys())[:3]))  # just show first few clip_ids for debugging
                continue

            # Process video
            try:
                cap = cv2.VideoCapture(video_path)
                frame_count = 0
                frame_sum = None

                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    gray = gray.astype(np.float32) / 255.0
                    if frame_sum is None:
                        frame_sum = gray
                    else:
                        frame_sum += gray
                    frame_count += 1

                cap.release()
                if frame_count > 0 and frame_sum is not None:
                    mean_frame = frame_sum / frame_count
                    video_features = mean_frame.flatten()
                    video_features_list.append(video_features)
                    video_labels.append(matched_emotion)

            except Exception as e:
                print(f"[VIDEO DEBUG] Error processing {video_file}: {e}")

    print(f"[VIDEO DEBUG] Matched: {matched_count}, Unmatched-labeled videos: {len(unmatched_list)}")
    if unmatched_list:
        print(f"[VIDEO DEBUG] Sample unmatched video files:")
        for video_file, tried_ids in unmatched_list[:5]:
            print(f"  Unmatched: {video_file} → Tried clip_ids starting with: {tried_ids}")

    return video_features_list, video_labels

print("Video preprocessing function (with prefix matching) loaded.")


Video preprocessing function (with prefix matching) loaded.


In [10]:
# ****************************************************************************************************
# BLOCK [5.4]: PREPROCESS MOTION CAPTURE (MoCap) IN BOTH "dialog" & "sentences"
# ****************************************************************************************************


def preprocess_motion_iemocap(data_path):
    """
    Searches each Session for 'dialog/MOCAP_*' and 'sentences/MOCAP_*' subdirs.
    Each MOCAP_* directory might contain .txt files with motion-capture data.
    We build a dictionary from EmoEvaluation to map a 'clip_id' -> emotion, and attempt
    to unify the .txt filenames with that 'clip_id' using a transform function.

    If the .txt is unmatched, we label it 'unknown'. Otherwise, we store the numeric features.
    Finally, we log how many total .txt are processed, how many matched, and how many unknown.
    """

    mocap_features_list = []
    mocap_labels = []

    total_mocap_files = 0
    matched_mocap_files = 0
    unknown_mocap_files = 0
    unmatched_info = []

    def transform_mocap_filename(base_name):
        """
        Example transform function for .txt filenames like "Ses01F_impro02_1" -> "Ses01F_impro02_F001".
        Adjust logic if your filenames differ significantly.
        """
        pattern = r"^(Ses\d+[MF]_(impro|script)\d+)(?:_(\w+))?$"
        m = re.match(pattern, base_name)
        if not m:
            # fallback: no transform
            return base_name

        prefix = m.group(1)  # e.g. "Ses01F_impro02"
        suffix = m.group(3)
        if suffix is None:
            return prefix + "_F000"
        elif suffix.isdigit():
            return f"{prefix}_F{suffix.zfill(3)}"
        else:
            # if letters exist, e.g. '1b'
            digits = "".join(ch for ch in suffix if ch.isdigit())
            if digits == "":
                return prefix + "_F000"
            else:
                return f"{prefix}_F{digits.zfill(3)}"

    # Loop sessions
    for session in range(1, 6):
        session_dir = os.path.join(data_path, f"Session{session}")
        emo_dir = os.path.join(session_dir, "dialog", "EmoEvaluation")
        
        if not os.path.isdir(session_dir):
            print(f"[MOCAP DEBUG] Session directory not found: {session_dir}")
            continue
        if not os.path.isdir(emo_dir):
            print(f"[MOCAP DEBUG] EmoEvaluation directory not found: {emo_dir}")
            continue

        # Build label dict: clip_id -> emotion
        mocap_label_dict = {}
        for emo_file in os.listdir(emo_dir):
            if emo_file.endswith(".txt"):
                with open(os.path.join(emo_dir, emo_file), "r") as ef:
                    for line in ef:
                        if line.startswith("[") and "]" in line:
                            parts = line.split("\t")
                            if len(parts) >= 3:
                                clip_id = parts[1].strip()
                                emotion = parts[2].strip()
                                mocap_label_dict[clip_id] = emotion

        # We now check both 'dialog' and 'sentences' subfolders for MOCAP_*
        for subfolder in ["dialog", "sentences"]:
            subfolder_path = os.path.join(session_dir, subfolder)
            if not os.path.isdir(subfolder_path):
                continue

            # For each item in subfolder, check if name starts with MOCAP
            for item in os.listdir(subfolder_path):
                if item.startswith("MOCAP"):
                    mocap_dir = os.path.join(subfolder_path, item)
                    if not os.path.isdir(mocap_dir):
                        continue
                    # Recursively find .txt in this MOCAP_* directory
                    for root, dirs, files in os.walk(mocap_dir):
                        for file_name in files:
                            if file_name.endswith(".txt"):
                                total_mocap_files += 1
                                full_path = os.path.join(root, file_name)
                                base_name = os.path.splitext(file_name)[0]

                                # Convert .txt filename -> possible clip_id
                                new_id = transform_mocap_filename(base_name)
                                # Lookup in dictionary
                                emotion_label = mocap_label_dict.get(new_id, "unknown")
                                if emotion_label == "unknown":
                                    unknown_mocap_files += 1
                                    unmatched_info.append((file_name, new_id))
                                else:
                                    matched_mocap_files += 1

                                # Attempt to read numeric data
                                try:
                                    data_array = np.loadtxt(full_path, delimiter=None)
                                    if data_array.ndim == 1:
                                        data_array = data_array.reshape(1, -1)
                                    mocap_feature = np.mean(data_array, axis=0)

                                    mocap_features_list.append(mocap_feature)
                                    mocap_labels.append(emotion_label)
                                except Exception as e:
                                    print(f"[MOCAP DEBUG] Error reading {full_path}: {e}")

    print(f"\n[MOCAP DEBUG] Found {total_mocap_files} total .txt files under MOCAP_* dirs in 'dialog'/'sentences'.")
    print(f"[MOCAP DEBUG] Matched: {matched_mocap_files}, Unknown: {unknown_mocap_files}")

    if unmatched_info:
        print("\n[MOCAP DEBUG] Some unmatched .txt files. Example:")
        for i, (fname, tried_id) in enumerate(unmatched_info[:5], 1):
            print(f"{i}) File: {fname} => tried clip_id: {tried_id}")

    return mocap_features_list, mocap_labels

print("BLOCK [5] loaded: Searches 'dialog/MOCAP_*' and 'sentences/MOCAP_*' for MoCap .txt files + label matching.")


BLOCK [5] loaded: Searches 'dialog/MOCAP_*' and 'sentences/MOCAP_*' for MoCap .txt files + label matching.


In [11]:
# ****************************************************************************************************
# BLOCK [6]: PREPROCESS ALL MODALITIES — QUIET VERSION (NO PRINTS)
# ****************************************************************************************************
import os
import io, contextlib, re
import numpy as np
import librosa
import cv2
import concurrent.futures

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# Helper to suppress console output while still running a function
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def _silent(fn, *args, **kwargs):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        return fn(*args, **kwargs)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 1) TEXT
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def preprocess_text_iemocap(data_path):
    from transformers import BertTokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    text_data, labels = [], []
    for session in range(1, 6):
        tdir = os.path.join(data_path, f"Session{session}", "dialog", "transcriptions")
        edir = os.path.join(data_path, f"Session{session}", "dialog", "EmoEvaluation")
        if not os.path.isdir(tdir) or not os.path.isdir(edir):
            continue
        for fn in os.listdir(tdir):
            if not fn.endswith(".txt") or fn.startswith("."):
                continue
            with open(os.path.join(tdir, fn)) as f:
                lines = f.readlines()
            emo_file = os.path.join(edir, fn)
            if not os.path.exists(emo_file):
                continue
            with open(emo_file) as ef:
                for line in ef:
                    if line.startswith("[") and "]" in line:
                        parts = line.split("\t")
                        if len(parts) >= 3 and lines:
                            labels.append(parts[2].strip())
                            text_data.append(lines[0].strip())
    return text_data, labels

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 2) AUDIO (parallel MFCC)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def process_wav_file(args):
    wav_path, lbl_dict = args
    base = os.path.splitext(os.path.basename(wav_path))[0]
    emo  = lbl_dict.get(base, "unknown")
    try:
        sig, sr = librosa.load(wav_path, sr=None)
        mfcc    = librosa.feature.mfcc(y=sig, sr=sr, n_mfcc=13)
        return np.mean(mfcc, axis=1), emo
    except:
        return None, None

def preprocess_audio_iemocap_parallel(data_path, n_workers=8):
    results, total = [], 0
    for s in range(1,6):
        wav_dir = os.path.join(data_path, f"Session{s}", "sentences", "wav")
        emo_dir = os.path.join(data_path, f"Session{s}", "dialog",    "EmoEvaluation")
        if not os.path.isdir(wav_dir) or not os.path.isdir(emo_dir):
            continue
        lbl_dict = {}
        for ef in os.listdir(emo_dir):
            if ef.endswith(".txt"):
                with open(os.path.join(emo_dir, ef)) as f:
                    for l in f:
                        if l.startswith("[") and "]" in l:
                            p = l.split("\t")
                            if len(p)>=3:
                                lbl_dict[p[1].strip()] = p[2].strip()
        wavs = []
        for root,_,files in os.walk(wav_dir):
            for fn in files:
                if fn.endswith(".wav"):
                    total += 1
                    wavs.append(os.path.join(root,fn))
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_workers) as ex:
            for feat, lab in ex.map(process_wav_file, [(w, lbl_dict) for w in wavs]):
                if feat is not None:
                    results.append((feat, lab))
    feats, labs = zip(*results) if results else ([],[])
    return list(feats), list(labs)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 3) VIDEO (prefix‐matching)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def preprocess_video_iemocap(data_path):
    feats, labs = [], []
    for s in range(1,6):
        vid_dir = os.path.join(data_path, f"Session{s}", "dialog", "avi", "DivX")
        emo_dir = os.path.join(data_path, f"Session{s}", "dialog", "EmoEvaluation")
        if not os.path.isdir(vid_dir) or not os.path.isdir(emo_dir):
            continue
        label_map = {}
        for ef in os.listdir(emo_dir):
            if ef.endswith(".txt"):
                with open(os.path.join(emo_dir, ef)) as f:
                    for l in f:
                        if l.startswith("[") and "]" in l:
                            p = l.split("\t")
                            if len(p)>=3:
                                label_map[p[1].strip()] = p[2].strip()
        for vf in os.listdir(vid_dir):
            if not (vf.endswith(".avi") or vf.endswith(".mpg")):
                continue
            base = os.path.splitext(vf)[0]
            emo  = next((e for cid,e in label_map.items() if cid.startswith(base)), None)
            if not emo:
                continue
            cap = cv2.VideoCapture(os.path.join(vid_dir, vf))
            ssum, cnt = None, 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)/255.0
                ssum = gray if ssum is None else ssum + gray
                cnt += 1
            cap.release()
            if cnt>0:
                feats.append((ssum/cnt).flatten())
                labs.append(emo)
    return feats, labs

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 4) MOTION
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def preprocess_motion_iemocap(data_path):
    feats, labs = [], []
    def xfn(b):
        m = re.match(r"^(Ses\d+[MF]_(impro|script)\d+)(?:_(\w+))?$", b)
        if not m: return b
        pr, su = m.group(1), m.group(3)
        if su is None:    return pr+"_F000"
        if su.isdigit():  return f"{pr}_F{int(su):03d}"
        digs = "".join(ch for ch in su if ch.isdigit())
        return f"{pr}_F{int(digs):03d}" if digs else pr+"_F000"
    for s in range(1,6):
        sd, ed = os.path.join(data_path,f"Session{s}"), os.path.join(data_path,f"Session{s}","dialog","EmoEvaluation")
        if not os.path.isdir(sd) or not os.path.isdir(ed):
            continue
        dmap = {}
        for ef in os.listdir(ed):
            if ef.endswith(".txt"):
                with open(os.path.join(ed,ef)) as f:
                    for l in f:
                        if l.startswith("[") and "]" in l:
                            p = l.split("\t")
                            if len(p)>=3:
                                dmap[p[1].strip()] = p[2].strip()
        for sub in ("dialog","sentences"):
            pth = os.path.join(sd,sub)
            if not os.path.isdir(pth): continue
            for itm in os.listdir(pth):
                if not itm.startswith("MOCAP"): continue
                for root,_,files in os.walk(os.path.join(pth,itm)):
                    for fn in files:
                        if fn.endswith(".txt"):
                            bid = os.path.splitext(fn)[0]
                            emo = dmap.get(xfn(bid), "unknown")
                            try:
                                arr = np.loadtxt(os.path.join(root,fn))
                                if arr.ndim==1: arr = arr.reshape(1,-1)
                                feats.append(arr.mean(axis=0)); labs.append(emo)
                            except:
                                pass
    return feats, labs

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 5) RUN SILENTLY
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
text_data,    text_labels    = _silent(preprocess_text_iemocap,           data_path)
audio_features_list, audio_labels   = _silent(preprocess_audio_iemocap_parallel, data_path, n_workers=8)
video_features_list, video_labels   = _silent(preprocess_video_iemocap,          data_path)
motion_features_list, motion_labels  = _silent(preprocess_motion_iemocap,         data_path)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 6) SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
print(f"✅ All four modalities preprocessed:")
print(f"  • text_data: {len(text_data)} samples")
print(f"  • audio_features_list: {len(audio_features_list)} samples")
print(f"  • video_features_list: {len(video_features_list)} samples")
print(f"  • motion_features_list: {len(motion_features_list)} samples")
print(f"  • text_labels: {len(text_labels)} labels")


KeyboardInterrupt: 

In [ ]:
# ****************************************************************************************************
# BLOCK [6A]:  BUILD AUDIO & VIDEO BACKBONES  +  FUSION MODEL   (auto-dims)
# ****************************************************************************************************
import torch
import torch.nn as nn

# --- auto-detect real feature sizes -------------------------------------------------
audio_feat_dim = len(audio_features_list[0])       # e.g. 13 MFCC dims
video_feat_dim = len(video_features_list[0])       # e.g. 345 600 flattened pixels

AUDIO_OUT_DIM  = 256
VIDEO_OUT_DIM  = 512
FUSION_DIM     = 256
NUM_CLASSES    = len(set(text_labels))              # quick count of unique labels

# --- tiny audio MLP -----------------------------------------------------------------
class AudioBackbone(nn.Module):
    def __init__(self, in_dim, out_dim=AUDIO_OUT_DIM):
        super().__init__()
        self.out_dim = out_dim
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 4 * out_dim),
            nn.ReLU(),
            nn.Linear(4 * out_dim, out_dim),
        )
    def forward(self, x):                           # [B,in_dim] or [B,1,in_dim]
        if x.dim() == 3: x = x.squeeze(1)
        return self.net(x)

# --- tiny video MLP -----------------------------------------------------------------
class VideoBackbone(nn.Module):
    def __init__(self, in_dim, out_dim=VIDEO_OUT_DIM):
        super().__init__()
        self.out_dim = out_dim
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 2 * out_dim),
            nn.ReLU(),
            nn.Linear(2 * out_dim, out_dim),
        )
    def forward(self, x):                           # [B,in_dim] or [B,1,in_dim]
        if x.dim() == 3: x = x.squeeze(1)
        return self.net(x)

audio_backbone = AudioBackbone(audio_feat_dim).to(device)
video_backbone = VideoBackbone(video_feat_dim).to(device)

print(f"✓ Audio backbone  : {audio_feat_dim}  →  {audio_backbone.out_dim}")
print(f"✓ Video backbone  : {video_feat_dim}  →  {video_backbone.out_dim}")

# --- fusion model -------------------------------------------------------------------
class AR_XMER_Model(nn.Module):
    def __init__(self, text_enc, audio_enc, video_enc,
                 fusion_dim=FUSION_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.text_enc  = text_enc
        self.audio_enc = audio_enc
        self.video_enc = video_enc
        self.fusion = nn.Sequential(
            nn.Linear(text_enc.out_dim +
                      audio_enc.out_dim +
                      video_enc.out_dim, fusion_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.classifier = nn.Linear(fusion_dim, num_classes)

    def forward(self, txt_ids, aud_vec, vid_vec):
        t = self.text_enc(txt_ids)
        a = self.audio_enc(aud_vec)
        v = self.video_enc(vid_vec)
        fused  = self.fusion(torch.cat([t, a, v], dim=1))
        return self.classifier(fused), fused

model = AR_XMER_Model(text_backbone, audio_backbone, video_backbone).to(device)
print("✅ Multimodal AR-XMER model built on", device)


In [ ]:
# ****************************************************************************************************
# BLOCK [6B]:  DEFINE FGSM / PGD ATTACKERS  (requires `model`)
# ****************************************************************************************************
import torchattacks

attack_fgsm = torchattacks.FGSM(model, eps=0.1)
attack_pgd  = torchattacks.PGD(model, eps=0.1, alpha=0.001, steps=10)

print("✓ FGSM and PGD attackers initialised.")


In [ ]:
# ****************************************************************************************************
# BLOCK [7]:  TRAIN ONE EPOCH (clean-only)
# ****************************************************************************************************
def train_one_epoch(model, dataloader, optimizer, criterion):
    """
    Batch = (ids, mask, aud, vid, moc, lbl)
    Ignores the motion tensor entirely.
    """
    model.train()
    running_loss = 0.0
    total        = 0

    for ids, mask, aud, vid, moc, lbl in dataloader:
        ids = ids.to(device)
        aud = aud.to(device)
        vid = vid.to(device)
        lbl = lbl.to(device)

        optimizer.zero_grad()
        logits, _ = model(ids, aud, vid)
        loss      = criterion(logits, lbl)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * lbl.size(0)
        total        += lbl.size(0)

    return running_loss / total


In [ ]:
# ****************************************************************************************************
# BLOCK [7.5]: MULTIMODAL DATASET & DATALOADER CREATION (with length-alignment & optional motion)
# ****************************************************************************************************

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer

# ————————— 1) Debug: Check that the preprocessing outputs are non-empty —————————
print("🔍 Preprocessed lengths:",
      "text_data =",      len(text_data),
      "audio_feats =",    len(audio_features_list),
      "video_feats =",    len(video_features_list),
      "motion_feats =",   len(motion_features_list),
      "text_labels =",    len(text_labels)
)

# ————————— 2) Figure out which modalities to include —————————
required = {
    "text":  len(text_data),
    "audio": len(audio_features_list),
    "video": len(video_features_list),
    "label": len(text_labels),
}
if len(motion_features_list) > 0:
    required["motion"] = len(motion_features_list)
    include_motion = True
else:
    print("⚠️ No motion features detected; proceeding WITHOUT motion modality.")
    include_motion = False

# ————————— 3) Compute common sample count & sanity-check —————————
n = min(required.values())
assert n > 0, f"No samples to load (n={n}) – check your preprocessing blocks!"

# ————————— 4) Fit the LabelEncoder on your first n labels —————————
le_mm = LabelEncoder().fit(text_labels[:n])

# ————————— 5) Load BERT tokenizer once —————————
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class ARXMERDataset(Dataset):
    def __init__(self,
                 texts,      # list of raw strings
                 audios,     # list of numpy arrays
                 videos,     # list of numpy arrays
                 motions,    # list of numpy arrays (may be empty)
                 labels,     # list of emotion strings
                 encoder,    # fitted LabelEncoder
                 tokenizer,  # BertTokenizer
                 max_len=128):
        self.n             = n
        self.include_motion= include_motion
        self.texts   = texts[:n]
        self.audios  = [torch.tensor(x, dtype=torch.float32) for x in audios[:n]]
        self.videos  = [torch.tensor(x, dtype=torch.float32) for x in videos[:n]]
        # if I have motion features, use them; otherwise create zero-vectors
        if include_motion:
            self.motions = [torch.tensor(x, dtype=torch.float32) for x in motions[:n]]
        else:
            # pick a shape that matches the video-feature dims; here we zero-pad to video.shape
            zero_shape = self.videos[0].shape
            self.motions = [torch.zeros(zero_shape, dtype=torch.float32) for _ in range(n)]

        self.labels  = torch.tensor(encoder.transform(labels[:n]), dtype=torch.long)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Tokenize on the fly
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].squeeze(0)      # [max_len]
        attention_mask = enc["attention_mask"].squeeze(0) # [max_len]

        return (
            input_ids,
            attention_mask,
            self.audios[idx],
            self.videos[idx],
            self.motions[idx],
            self.labels[idx],
        )

# ————————— 6) Instantiate the full dataset & split —————————
full_ds = ARXMERDataset(
    texts     = text_data,
    audios    = audio_features_list,
    videos    = video_features_list,
    motions   = motion_features_list,
    labels    = text_labels,
    encoder   = le_mm,
    tokenizer = tokenizer,
)

train_n = int(0.8 * len(full_ds))
val_n   = len(full_ds) - train_n
train_ds, val_ds = random_split(full_ds, [train_n, val_n])

batch_size_mm = 64
train_loader  = DataLoader(train_ds, batch_size=batch_size_mm, shuffle=True,  num_workers=0)
val_loader    = DataLoader(val_ds,   batch_size=batch_size_mm, shuffle=False, num_workers=0)

print(f"✅ train_loader: {len(train_ds)} samples, val_loader: {len(val_ds)} samples")


In [ ]:
# ****************************************************************************************************
# BLOCK [8] : TRAINING CONFIG — epochs & safety knobs for long runs
# ****************************************************************************************************
import torch, random, os, numpy as np

# ---- epochs (raise these as needed) ---------------------------------------------------------------
EPOCHS_MAIN = 100000       # was 5; main multimodal training
EPOCHS_MLP  = 100000       # stage-2 / head fine-tune (block [12])

# ---- optim/safety --------------------------------------------------------------------------------
USE_AMP         = True      # mixed precision to lower GPU mem
GRAD_CLIP_NORM  = 1.0       # clip to stabilize long runs (0 or None to disable)
VAL_EVERY       = 1         # validate every epoch
CKPT_EVERY      = 5         # save checkpoint every N epochs
LOG_EVERY_STEPS = 50        # print every N steps

# ---- reproducibility -----------------------------------------------------------------------------
SEED = 2025
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True  # better throughput, ok for long runs
seed_everything()

# ---- tiny helper: safe unpack (works even if batch has extra tensors) ----------------------------
def _unpack_batch_for_training(batch, audio_feat_dim=13, video_feat_dim=345600):
    ids = aud = vid = lbl = None
    for x in batch:
        if not isinstance(x, torch.Tensor): 
            continue
        if ids is None and x.dtype == torch.long and x.dim() == 2:
            ids = x; continue
        if aud is None and x.dtype.is_floating_point and x.dim()==2 and x.shape[-1]==audio_feat_dim:
            aud = x; continue
        if vid is None and x.dtype.is_floating_point and x.dim()==2 and x.shape[-1]==video_feat_dim:
            vid = x; continue
        if lbl is None and x.dtype == torch.long and x.dim()==1:
            lbl = x
    if ids is None or aud is None or vid is None or lbl is None:
        shapes = [tuple(x.shape) for x in batch if isinstance(x, torch.Tensor)]
        raise RuntimeError(f"[train] Could not unpack batch; saw shapes: {shapes}")
    return ids, aud, vid, lbl

print(f"[8] Config — epochs: main={EPOCHS_MAIN}, mlp={EPOCHS_MLP}, amp={USE_AMP}, clip={GRAD_CLIP_NORM}")
# ****************************************************************************************************


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# BLOCK [8.5]: VALIDATION FUNCTION
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def validate(model, dataloader, criterion):
    """
    Returns (avg_loss, accuracy) on the provided dataloader.
    Batch = (ids, mask, aud, vid, moc, lbl)
    """
    model.eval()
    total_loss = 0.0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for ids, mask, aud, vid, moc, lbl in dataloader:
            ids = ids.to(device)
            aud = aud.to(device)
            vid = vid.to(device)
            lbl = lbl.to(device)

            logits, _ = model(ids, aud, vid)
            loss      = criterion(logits, lbl)

            total_loss += loss.item() * lbl.size(0)
            correct    += (logits.argmax(1) == lbl).sum().item()
            total      += lbl.size(0)

    return total_loss / total, correct / total


In [ ]:
# ****************************************************************************************
# BLOCK [8.6] : CHECKPOINT / LOGGING CONFIG  — idempotent, safe to re-run
# ****************************************************************************************
import os, json, glob, torch

# 1) Where to store checkpoints
#    Prefer $CKPT_DIR, else $SCRATCH (cluster scratch), else a local folder.
CKPT_DIR = os.environ.get("CKPT_DIR")
if not CKPT_DIR:
    base = os.environ.get("SCRATCH", os.getcwd())
    CKPT_DIR = os.path.join(base, "AR_XMER_ckpts")
os.makedirs(CKPT_DIR, exist_ok=True)

# 2) Retention policy (can be overridden by env vars)
CKPT_KEEP_LAST = int(os.environ.get("CKPT_KEEP_LAST", 2))   # keep the most recent 'last' + a couple epoch files
CKPT_KEEP_BEST = int(os.environ.get("CKPT_KEEP_BEST", 1))   # keep N best snapshots
CKPT_EVERY     = int(os.environ.get("CKPT_EVERY",    50))   # save an epoch file only every N epochs
CKPT_MAX_BYTES = int(os.environ.get("CKPT_MAX_BYTES", 8 * 1024**3))  # 8 GiB budget

META_FILE = os.path.join(CKPT_DIR, "meta.json")
_state = {"best_val": float("inf"), "best_path": None}
if os.path.isfile(META_FILE):
    try:
        _state.update(json.load(open(META_FILE)))
    except Exception:
        pass

def _dir_bytes(path):
    total = 0
    for f in glob.glob(os.path.join(path, "*.pth")):
        try: total += os.path.getsize(f)
        except: pass
    return total

def _atomic_save(obj, tag):
    tmp   = os.path.join(CKPT_DIR, f"{tag}.tmp")
    final = os.path.join(CKPT_DIR, f"{tag}.pth")
    torch.save(obj, tmp)
    os.replace(tmp, final)  # atomic
    return final

def _rotate_space():
    """Enforce space budget and retention (never delete best/last)."""
    protected = {os.path.join(CKPT_DIR, "best.pth"),
                 os.path.join(CKPT_DIR, "last.pth")}
    # Keep only last K epoch files by epoch number in filename
    epoch_files = sorted(glob.glob(os.path.join(CKPT_DIR, "epoch*.pth")))
    if CKPT_KEEP_LAST >= 0 and len(epoch_files) > CKPT_KEEP_LAST:
        for f in epoch_files[:-CKPT_KEEP_LAST]:
            if f not in protected:
                try: os.remove(f)
                except: pass

    # Enforce overall size budget
    while _dir_bytes(CKPT_DIR) > CKPT_MAX_BYTES:
        # delete oldest non-protected file
        cand = sorted(
            [f for f in glob.glob(os.path.join(CKPT_DIR, "*.pth")) if f not in protected],
            key=os.path.getmtime
        )
        if not cand: break
        try: os.remove(cand[0])
        except: break

def save_last(model, optimizer, epoch):
    payload = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": int(epoch)}
    path = _atomic_save(payload, "last")
    _rotate_space()
    return path

def maybe_save_best(val_loss, model, optimizer, epoch):
    """Save best snapshot if val_loss improved; returns True if saved."""
    global _state
    if val_loss < _state["best_val"]:
        _state["best_val"] = float(val_loss)
        _state["best_path"] = _atomic_save(
            {"model": model.state_dict(),
             "optimizer": optimizer.state_dict(),
             "epoch": int(epoch),
             "val_loss": float(val_loss)},
            "best"
        )
        try: json.dump(_state, open(META_FILE, "w"))
        except: pass
        _rotate_space()
        return True
    return False

def save_epoch(model, optimizer, epoch):
    """Occasional epoch snapshot (will be auto-rotated)."""
    tag = f"epoch{int(epoch):04d}"
    _atomic_save({"model": model.state_dict(),
                  "optimizer": optimizer.state_dict(),
                  "epoch": int(epoch)}, tag)
    _rotate_space()


In [ ]:
# =============================================================================
# BLOCK [8.7]: CHECKPOINT HELPERS (safe to re-run)
# =============================================================================
import os, gc, json, math, shutil
from pathlib import Path

def _env_int(name, default):
    try:
        return int(os.environ.get(name, str(default)))
    except Exception:
        return default

# Where to save & how many to keep (env vars override during SLURM runs)
CKPT_DIR   = Path(os.environ.get("CKPT_DIR", Path.cwd() / "checkpoints"))
CKPT_EVERY = _env_int("CKPT_EVERY", 50)   # save every N epochs
CKPT_KEEP  = _env_int("CKPT_KEEP",   2)   # keep only last N epoch checkpoints

CKPT_DIR.mkdir(parents=True, exist_ok=True)

class SaveManager:
    """Rotate epoch checkpoints; 'best.pt' is always overwritten (no growth)."""
    def __init__(self, ckpt_dir: Path, keep: int = 2, prefix: str = "ar_xmer"):
        self.dir    = Path(ckpt_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.keep   = max(0, int(keep))
        self.prefix = prefix
        self._ring  = []  # paths we keep rotating

    def _prune(self):
        excess = len(self._ring) - self.keep
        for _ in range(max(0, excess)):
            old = self._ring.pop(0)
            try: old.unlink(missing_ok=True)
            except Exception: pass

    def save_epoch(self, epoch:int, payload:dict):
        tmp   = self.dir / f"{self.prefix}_epoch{epoch:04d}.tmp.pt"
        final = self.dir / f"{self.prefix}_epoch{epoch:04d}.pt"
        torch.save(payload, tmp)
        tmp.replace(final)            # atomic rename
        self._ring.append(final)
        self._prune()

    def save_best(self, payload:dict):
        tmp   = self.dir / f"{self.prefix}_best.tmp.pt"
        final = self.dir / f"{self.prefix}_best.pt"
        torch.save(payload, tmp)
        tmp.replace(final)            # overwrite; no growth


In [ ]:
# ======================================================================================
# BLOCK [8.9]: STORAGE & CACHE PATHS (run-once setup; keeps files off HOME quota)
# ======================================================================================
import os, pathlib, re, gc, datetime, torch

USER     = os.environ.get("USER", "user")
SCRATCH = (os.environ.get("SCRATCH")
           or os.environ.get("SLURM_TMPDIR")
           or os.environ.get("TMPDIR")
           or f"/groups/hariri/{USER}")


RUN_NAME = os.environ.get("RUN_NAME", "arxmer")

BASE_DIR = pathlib.Path.cwd()
CKPT_DIR = pathlib.Path(os.environ.get("CKPT_DIR", f"{SCRATCH}/{RUN_NAME}/checkpoints")).expanduser()
LOG_DIR  = pathlib.Path(os.environ.get("LOG_DIR",  f"{SCRATCH}/{RUN_NAME}/logs")).expanduser()
TMP_DIR  = pathlib.Path(os.environ.get("TMP_DIR",  f"{SCRATCH}/{RUN_NAME}/tmp")).expanduser()

for d in (CKPT_DIR, LOG_DIR, TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Send all heavy caches to SCRATCH (not HOME)
os.environ.update({
    "TORCH_HOME":           f"{SCRATCH}/{RUN_NAME}/torch",
    "XDG_CACHE_HOME":       f"{SCRATCH}/{RUN_NAME}/.cache",
    "TRANSFORMERS_CACHE":   f"{SCRATCH}/{RUN_NAME}/hf",
    "HF_HOME":              f"{SCRATCH}/{RUN_NAME}/hf",
    "HF_DATASETS_CACHE":    f"{SCRATCH}/{RUN_NAME}/hf_datasets",
    "MPLCONFIGDIR":         f"{SCRATCH}/{RUN_NAME}/mpl",
    "TMPDIR":               str(TMP_DIR),
})

# ---- lean checkpoint policy (used by BLOCK [9]) ---------------------------------------
CKPT_KEEP        = 2        # keep newest N epoch ckpts if you ever enable them
CKPT_EVERY       = 0        # 0 = don't save per-epoch ckpts
SAVE_BEST_ONLY   = True     # only best model (val loss) as 'best.pth'
SAVE_HEAD_ONLY   = False    # True => save only fusion/classifier layers
SAVE_HALF        = True     # save fp16 to shrink file size

def _shrink_state_dict(state, head_only=False, save_half=True):
    if head_only:
        wanted = tuple(["fusion.", "classifier."])
        state = {k: v for k, v in state.items() if k.startswith(wanted)}
    out = {}
    for k, v in state.items():
        if torch.is_tensor(v):
            t = v.detach().cpu()
            if save_half and t.is_floating_point():
                try: t = t.half()
                except Exception: pass
            out[k] = t
        else:
            out[k] = v
    return out

def save_best_model(model, best_loss, current_loss):
    if current_loss <= best_loss:
        state = _shrink_state_dict(model.state_dict(),
                                   head_only=SAVE_HEAD_ONLY,
                                   save_half=SAVE_HALF)
        torch.save({"model": state, "val_loss": float(current_loss)},
                   CKPT_DIR / "best.pth")
        return current_loss, True
    return best_loss, False

def rotate_ckpts(keep=CKPT_KEEP, pattern=r"^ckpt_epoch\d+\.pth$"):
    files = sorted([p for p in CKPT_DIR.iterdir() if re.match(pattern, p.name)],
                   key=lambda x: x.stat().st_mtime, reverse=True)
    for p in files[keep:]:
        try: p.unlink()
        except FileNotFoundError:
            pass


In [ ]:
# ****************************************************************************************************
# BLOCK [9]: TRAIN/EVAL EPOCH — FULL REPLACEMENT (no tensor truthiness)
# ****************************************************************************************************
import torch
import torch.nn.functional as F

# Pull common knobs if they already exist elsewhere; fall back to safe defaults
AMP        = bool(globals().get("amp", globals().get("USE_AMP", True)))
CLIP_NORM  = float(globals().get("clip", globals().get("CLIP_NORM", 1.0)))
scaler     = globals().get("scaler") or torch.cuda.amp.GradScaler(enabled=AMP)

def _to_device(x):
    return x.to(device, non_blocking=True) if isinstance(x, torch.Tensor) else x

def _extract_modalities(batch):
    """
    Expected batch layout used in AR-XMER:
      batch[0] -> text ids (or dict/tuple for text backbone)
      batch[2] -> audio features tensor (may be absent or empty)
      batch[3] -> video features tensor (may be absent or empty)
      batch[-1] -> labels
    We treat missing/empty tensors as None (never use Python truthiness on tensors).
    """
    ids = batch[0]
    lbl = batch[-1]

    aud = None
    if len(batch) > 2 and torch.is_tensor(batch[2]) and batch[2].numel() > 0:
        aud = batch[2]

    vid = None
    if len(batch) > 3 and torch.is_tensor(batch[3]) and batch[3].numel() > 0:
        vid = batch[3]

    return ids, aud, vid, lbl

def _run_epoch(train: bool):
    """Run one epoch and return (avg_loss, accuracy)."""
    dl = train_loader if train else val_loader
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in dl:
        ids, aud, vid, lbl = _extract_modalities(batch)

        # Move to device safely (no implicit boolean checks)
        ids = _to_device(ids)
        lbl = _to_device(lbl)
        if aud is not None:
            aud = _to_device(aud)
        if vid is not None:
            vid = _to_device(vid)

        optimizer.zero_grad(set_to_none=True)

        if train:
            with torch.cuda.amp.autocast(enabled=AMP):
                logits, _ = model(ids, aud, vid)
                loss = F.cross_entropy(logits, lbl)

            scaler.scale(loss).backward()
            if CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), torch.cuda.amp.autocast(enabled=AMP):
                logits, _ = model(ids, aud, vid)
                loss = F.cross_entropy(logits, lbl)

        # Stats (be explicit; do not use tensors as booleans)
        bsz = int(lbl.size(0)) if isinstance(lbl, torch.Tensor) and lbl.ndim >= 1 else 1
        total_loss += float(loss.detach().item()) * bsz
        total_correct += (logits.argmax(1) == lbl).sum().item()
        total_count += bsz

        # Cleanup
        del ids, lbl, aud, vid, logits, loss
        torch.cuda.empty_cache()

    avg_loss = total_loss / max(1, total_count)
    acc = total_correct / max(1, total_count)
    return avg_loss, acc


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# BLOCK [9.5]: PREPARE TEXT TENSORS FOR FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
import torch

# train_ds and val_ds are torch.utils.data.Subset instances of ARXMERDataset
# Each item: (input_ids, attention_mask, audio, video, motion, label)

# Stack input_ids and attention_masks
X_train_ids  = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
X_train_attn = torch.stack([train_ds[i][1] for i in range(len(train_ds))])
X_val_ids    = torch.stack([val_ds[i][0]   for i in range(len(val_ds))])
X_val_attn   = torch.stack([val_ds[i][1]   for i in range(len(val_ds))])


In [ ]:
# ****************************************************************************************************
# BLOCK [10]: EXTRACT TEXT FEATURES WITH BERT
# ****************************************************************************************************

import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA
from transformers import BertForSequenceClassification

# 1) instantiate a classification‐head BERT (or load the fine-tuned weights)
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(le_mm.classes_)     # same LabelEncoder from Block 7.5
).to(device)

def extract_text_features(bert_model, input_ids, attention_masks, device, batch_size=64):
    """
    Return softmax-normalised logits for each sample.
    """
    bert_model.eval()
    ds     = TensorDataset(input_ids, attention_masks)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    feats_all = []
    with torch.no_grad():
        for b_ids, b_attn in loader:
            b_ids  = b_ids.to(device)
            b_attn = b_attn.to(device)
            logits = bert_model(b_ids, attention_mask=b_attn).logits
            feats  = torch.nn.functional.softmax(logits, dim=1)
            feats_all.append(feats.cpu().numpy())
    return np.concatenate(feats_all, axis=0)

# 2) Extract features for TRAIN and VAL sets created in Block 9.5
X_train_text_features = extract_text_features(
    bert_model,
    X_train_ids,
    X_train_attn,
    device
)
X_val_text_features = extract_text_features(
    bert_model,
    X_val_ids,
    X_val_attn,
    device
)

# 3) Dimensionality reduction (optional)
pca_text = PCA(n_components=0.95)
X_train_text_features_pca = pca_text.fit_transform(X_train_text_features)
X_val_text_features_pca   = pca_text.transform(X_val_text_features)

print(f"Text feature shape after PCA: {X_train_text_features_pca.shape}")


In [ ]:
# ****************************************************************************************************
# BLOCK [11] – EXPLANATION GENERATION & ERI CALCULATION  (memory-friendly, 1-sample version)
# ****************************************************************************************************
import torch
from captum.attr import IntegratedGradients

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 1)  Saliency helper
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def compute_saliency(model,
                     txt_ids,         # [1, L] LongTensor
                     aud_vec,         # [1, A]  or [1,1,A]  FloatTensor
                     vid_vec,         # [1, V]  or [1,1,V]  FloatTensor
                     target_class: int,
                     n_steps: int = 4):
    """
    Returns a tuple (attr_text, attr_audio, attr_video) for ONE sample.
    We:
      • keep the genuine gradients for audio & video;
      • return a zero-tensor for text (token IDs are non-differentiable);
      • use a tiny `n_steps` + `internal_batch_size=1` to avoid OOM.
    """
    model.eval()

    # ---------- audio attribution --------------------------------------------------
    ig_a = IntegratedGradients(lambda a: model(txt_ids, a, vid_vec)[0])
    attr_audio = ig_a.attribute(
        inputs=aud_vec,
        baselines=torch.zeros_like(aud_vec),
        target=target_class,
        n_steps=n_steps,
        internal_batch_size=1,
        return_convergence_delta=False
    )

    # ---------- video attribution --------------------------------------------------
    ig_v = IntegratedGradients(lambda v: model(txt_ids, aud_vec, v)[0])
    attr_video = ig_v.attribute(
        inputs=vid_vec,
        baselines=torch.zeros_like(vid_vec),
        target=target_class,
        n_steps=n_steps,
        internal_batch_size=1,
        return_convergence_delta=False
    )

    # ---------- text placeholder (0-attribution) -----------------------------------
    attr_text = torch.zeros_like(txt_ids, dtype=torch.float32)

    return (attr_text, attr_audio, attr_video)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 2)  ERI helper
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
def explanation_robustness_index(attr_clean, attr_defended):
    """
    Intersection-over-Union on thresholded saliency masks, then averaged
    over the three modalities.
    """
    ious = []
    for ac, ad in zip(attr_clean, attr_defended):
        m_c = (ac.detach().cpu().abs() > 1e-5).float().view(-1)
        m_d = (ad.detach().cpu().abs() > 1e-5).float().view(-1)
        inter = (m_c * m_d).sum()
        union = m_c.sum() + m_d.sum() - inter + 1e-8
        ious.append((inter / union).item())
    return sum(ious) / len(ious)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# 3)  Tiny sanity-check on **one** validation example
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
model.eval()
ids, attn, aud, vid, moc, lbl = [x.to(device) for x in next(iter(val_loader))]
# Slice the FIRST sample only ↓↓↓
ids1, aud1, vid1, lbl1 = ids[:1], aud[:1], vid[:1], lbl[0:1]

# Forward pass → prediction
logits1, _ = model(ids1, aud1, vid1)
pred1 = logits1.argmax(dim=1).item()

# Clean saliency
attr_clean = compute_saliency(model, ids1, aud1, vid1, target_class=pred1)

# Dummy “defended” saliency: jitter the audio slightly (5 % white noise)
aud1_adv = aud1 + 0.05 * torch.randn_like(aud1)
attr_def = compute_saliency(model, ids1, aud1_adv, vid1, target_class=pred1)

eri = explanation_robustness_index(attr_clean, attr_def)
print(f"✓ Sample ERI (clean vs noisy-audio) = {eri:.4f}")
torch.cuda.empty_cache()


In [ ]:
# ****************************************************************************************
# BLOCK [12] : HEAD-ONLY FINE-TUNING (classifier MLP) — AMP + grad-clip + safe scheduler
# ****************************************************************************************
import math, gc, torch
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR

# ---- hyperparams / globals with safe defaults ------------------------------------------
EPOCHS_MLP       = globals().get("EPOCHS_MLP", 100000)
lr_mlp           = globals().get("lr_mlp", 1e-4)
USE_AMP          = globals().get("USE_AMP", True)
GRAD_CLIP_NORM   = globals().get("GRAD_CLIP_NORM", 1.0)
VAL_EVERY_HEAD   = globals().get("VAL_EVERY_HEAD", 1)

assert "train_loader" in globals() and "val_loader" in globals(), "Loaders not found."
assert "model" in globals() and "device" in globals(), "Model/device not set."

# ---- loss -------------------------------------------------------------------------------
criterion = torch.nn.CrossEntropyLoss()

# ---- locate the head (prefer `model.classifier`, else last Linear as fallback) ----------
head = getattr(model, "classifier", None)
if head is None:
    head = None
    for m in reversed(list(model.modules())):
        if isinstance(m, torch.nn.Linear):
            head = m
            break
    assert head is not None, "Could not locate classifier head in model."

# ---- freeze everything except head ------------------------------------------------------
for p in model.parameters():
    p.requires_grad = False
for p in head.parameters():
    p.requires_grad = True
model.train()

# ---- optimizer + scheduler + AMP scaler ------------------------------------------------
optimizer_mlp  = torch.optim.AdamW(head.parameters(), lr=lr_mlp, weight_decay=0.0)
scheduler_mlp  = StepLR(optimizer_mlp, step_size=max(1, EPOCHS_MLP // 3), gamma=0.5)
scaler_mlp     = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ---- one epoch helper ------------------------------------------------------------------
def _run_epoch_head(epoch: int, train: bool):
    dl = train_loader if train else val_loader
    model.train() if train else model.eval()

    tot_loss, tot_ok, tot_seen = 0.0, 0, 0
    for step, batch in enumerate(dl, 1):
        # robust unpack: ids at 0, label at -1; find audio/video by shape
        lbl = batch[-1].to(device)
        ids = batch[0].to(device)

        aud, vid = None, None
        for x in batch[1:-1]:
            if not torch.is_tensor(x):
                continue
            if x.dim() >= 2 and (x.shape[-1] == 13 or (x.dim() > 2 and x.shape[1] == 13)):
                aud = x
            elif x.dim() == 2 and x.shape[-1] >= 200_000:
                vid = x
        if aud is None: aud = batch[2]
        if vid is None: vid = batch[3]

        aud = aud.to(device)
        vid = vid.to(device)

        if train:
            optimizer_mlp.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits, _ = model(ids, aud, vid)
                loss = criterion(logits, lbl)
            scaler_mlp.scale(loss).backward()
            if GRAD_CLIP_NORM:
                torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP_NORM)
            scaler_mlp.step(optimizer_mlp)
            scaler_mlp.update()
        else:
            with torch.no_grad(), torch.cuda.amp.autocast(enabled=USE_AMP):
                logits, _ = model(ids, aud, vid)
                loss = criterion(logits, lbl)

        bsz = ids.size(0)
        tot_loss += loss.item() * bsz
        tot_ok   += (logits.argmax(1) == lbl).sum().item()
        tot_seen += bsz

        del ids, aud, vid, lbl, logits, loss
        if step % 20 == 0:
            torch.cuda.empty_cache()

    avg_loss = tot_loss / max(1, tot_seen)
    avg_acc  = tot_ok  / max(1, tot_seen)
    return avg_loss, avg_acc

# ---- train loop ------------------------------------------------------------------------
best_val_head = -1.0
for epoch in range(1, EPOCHS_MLP + 1):
    gc.collect(); torch.cuda.empty_cache()

    tr_loss, tr_acc = _run_epoch_head(epoch, train=True)
    val_loss, val_acc = _run_epoch_head(epoch, train=False)

    print(f"[12] epoch {epoch:03d} | train_loss={tr_loss:.4f} acc={tr_acc:.3f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.3f}")

    if val_acc > best_val_head:
        best_val_head = val_acc
        torch.save({
            "model": model.state_dict(),
            "optimizer_mlp": optimizer_mlp.state_dict(),
            "epoch": epoch
        }, f"ckpt_head_best_epoch{epoch:03d}.pth")

    scheduler_mlp.step()

gc.collect(); torch.cuda.empty_cache()

# ---- unfreeze whole model so subsequent blocks (e.g., [17-B], [19], [20]) work as usual -
for p in model.parameters():
    p.requires_grad = True
print("[12] Head fine-tune finished.")


In [ ]:
# ****************************************************************************************************
# BLOCK [13]: SAVE PAIRED CLEAN / ADVERSARIAL SAMPLES FOR IEMOCAP-Adv
# ****************************************************************************************************
import os
import numpy as np
import soundfile as sf
import torch
from torchvision.transforms import ToPILImage

def save_adversarial_pairs(model, dataset, save_dir, attack_fn, num_examples=100):
    """
    Iterates over `dataset` (a Subset of ARXMERDataset), generates one adversarial
    version per modality, and saves:
      • Text token IDs  → .txt
      • Audio waveforms  → .wav
      • Video feature vectors → .npy
    """
    os.makedirs(save_dir, exist_ok=True)
    count = 0

    for i, sample in enumerate(dataset):
        if count >= num_examples:
            break

        # ───────── Unpack exactly what ARXMERDataset returns: ─────────
        #   (input_ids, attention_mask, audio_vec, video_vec, motion_vec, label)
        input_ids, attention_mask, audio_vec, video_vec, motion_vec, label = sample

        # Add batch‐dim and move to GPU
        it = input_ids.unsqueeze(0).to(device)
        ia = audio_vec.unsqueeze(0).to(device)
        iv = video_vec.unsqueeze(0).to(device)

        # ───────── Clean prediction ─────────
        model.eval()
        with torch.no_grad():
            logits_clean, _ = model(it, ia, iv)
        pred = logits_clean.argmax(dim=1)

        # ───────── Adversarial examples ─────────
        # Text attack often needs special handling; here I simply skip it
        adv_it = it
        # Audio
        try:
            adv_ia = attack_fn(ia, pred)
        except Exception:
            adv_ia = ia
        # Video
        try:
            adv_iv = attack_fn(iv, pred)
        except Exception:
            adv_iv = iv

        # ───────── Save TEXT token IDs ─────────
        txt_clean = it.cpu().squeeze(0).tolist()
        txt_adv   = adv_it.cpu().squeeze(0).tolist()
        with open(f"{save_dir}/sample_{i}_text_clean.txt", "w") as f:
            f.write(" ".join(map(str, txt_clean)))
        with open(f"{save_dir}/sample_{i}_text_adv.txt", "w") as f:
            f.write(" ".join(map(str, txt_adv)))

        # ───────── Save AUDIO as .wav ─────────
        # assuming `audio_vec` is a 1D waveform; adjust samplerate as needed
        audio_clean = ia.cpu().squeeze(0).numpy()
        audio_adv   = adv_ia.cpu().squeeze(0).numpy()
        sf.write(f"{save_dir}/sample_{i}_audio_clean.wav", audio_clean, samplerate=16000)
        sf.write(f"{save_dir}/sample_{i}_audio_adv.wav",   audio_adv,   samplerate=16000)

        # ───────── Save VIDEO features as .npy ─────────
        # these are the flattened mean frames
        np.save(f"{save_dir}/sample_{i}_video_clean.npy", iv.cpu().squeeze(0).numpy())
        np.save(f"{save_dir}/sample_{i}_video_adv.npy",   adv_iv.cpu().squeeze(0).numpy())

        count += 1

    print(f"Saved {count} clean/adv sample pairs under {save_dir}.")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# To run:

save_adversarial_pairs(
    model,
    val_loader.dataset,
    save_dir="iemocap_adv",
    attack_fn=attack_pgd,
    num_examples=50
)





# ****************************************************************************************************


In [ ]:
# ****************************************************************************************************
# BLOCK [14]: FILTER VALID AUDIO SAMPLES
# ****************************************************************************************************
# Remove 'unknown' or mismatched labels before using audio features in ML pipeline
valid_audio_indices = [i for i, label in enumerate(audio_labels) if label != "unknown" and audio_features_list[i] is not None]

audio_features_clean = [audio_features_list[i] for i in valid_audio_indices]
audio_labels_clean = [audio_labels[i] for i in valid_audio_indices]

print(f"Valid audio samples: {len(audio_features_clean)}")


In [ ]:
# ****************************************************************************************************
# BLOCK [15]: FINAL EVALUATION & METRICS REPORT  
# ****************************************************************************************************

eps = 0.1   # FGSM budget; tweak as desired

# 1) Clean baseline via helper
val_loss, clean_acc = validate(model, val_loader, criterion)
print(f"Validation Loss: {val_loss:.4f}  |  Clean Acc: {clean_acc:.2%}")

adv_acc_scores = []
eri_scores     = []

for ids, mask, aud, vid, moc, lbl in val_loader:
    # move to device
    ids, aud, vid, lbl = (x.to(device) for x in (ids, aud, vid, lbl))

    # ---- 1) Clean forward ----
    model.eval()
    with torch.no_grad():
        logits_clean, _ = model(ids, aud, vid)
        preds_clean    = logits_clean.argmax(dim=1)

    # ---- 2) FGSM on audio ----
    aud_adv = aud.clone().detach().requires_grad_(True)
    model.zero_grad()
    logits_a, _ = model(ids, aud_adv, vid)
    loss_a      = criterion(logits_a, preds_clean)
    loss_a.backward()
    adv_a       = (aud_adv + eps * aud_adv.grad.sign()).detach()

    # ---- 3) FGSM on video ----
    vid_adv = vid.clone().detach().requires_grad_(True)
    model.zero_grad()
    logits_v, _ = model(ids, aud, vid_adv)
    loss_v      = criterion(logits_v, preds_clean)
    loss_v.backward()
    adv_v       = (vid_adv + eps * vid_adv.grad.sign()).detach()

    # ---- 4) Discrete text left intact ----
    adv_t = ids

    # ---- 5) Adversarial predictions ----
    logits_adv_t, _ = model(adv_t,   aud,     vid)
    logits_adv_a, _ = model(ids,     adv_a,   vid)
    logits_adv_v, _ = model(ids,     aud,     adv_v)

    stacked = torch.stack([
        logits_adv_t.argmax(dim=1),
        logits_adv_a.argmax(dim=1),
        logits_adv_v.argmax(dim=1),
    ], dim=0)

    # majority vote
    adv_mode = torch.mode(stacked, dim=0)[0]
    adv_acc_scores.append((adv_mode == lbl).float().mean().item())

    # ---- 6) ERI on first sample of this batch ----
    #   clean saliency
    a_clean = compute_saliency(
        model,
        ids[0:1], aud[0:1], vid[0:1],
        target_class=preds_clean[0]
    )
    # “defended” saliency (text unchanged)
    a_adv = compute_saliency(
        model,
        adv_t[0:1], aud[0:1], vid[0:1],
        target_class=preds_clean[0]
    )
    eri_scores.append(explanation_robustness_index(a_clean, a_adv))

# 7) Summarize
overall_adv_acc = sum(adv_acc_scores) / len(adv_acc_scores)
overall_eri     = sum(eri_scores)     / len(eri_scores)

print(f"Adversarial Acc (majority vote): {overall_adv_acc:.2%}")
print(f"Average ERI over batches        : {overall_eri:.4f}")


In [ ]:
# ****************************************************************************************************
# BLOCK [16]: TRAIN & EVALUATE MLP ON VIDEO FEATURES  (CPU-safe, no GPU memory use)
# ****************************************************************************************************
import gc, numpy as np
from sklearn.model_selection   import train_test_split
from sklearn.metrics           import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.multiclass  import unique_labels

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---- make sure GPU cache isn't holding memory from previous cells (optional) ------------------------
if torch.cuda.is_available():
    try:
        torch.cuda.synchronize()
    except Exception:
        pass
    torch.cuda.empty_cache()

# ----------------- MLP head -------------------------------------------------------------------------
class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int, p_drop: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden_dim, n_classes),
        )
    def forward(self, x):
        return self.net(x)

# ----------------- hyperparams (fallbacks if not defined earlier) -----------------------------------
hidden_dim      = int(globals().get("hidden_dim", 512))
epochs_mlp      = int(globals().get("epochs_mlp", 100000))
batch_size_mlp  = int(globals().get("batch_size_mlp", 64))
lr_mlp          = float(globals().get("lr_mlp", 1e-6))
num_classes     = int(globals().get("num_classes", len(le_mm.classes_)))

# ----------------- 1) filter invalid labels ---------------------------------------------------------
valid_idx          = [i for i, lbl in enumerate(video_labels) if lbl != "unknown"]
video_feats_clean  = [video_features_list[i] for i in valid_idx]
video_labs_clean   = [video_labels[i]        for i in valid_idx]

# ----------------- 2) arrays + encode labels --------------------------------------------------------
X_v = np.vstack(video_feats_clean).astype("float32")
y_v = le_mm.transform(video_labs_clean).astype("int64")

# ----------------- 3) split (stratify if possible) --------------------------------------------------
try:
    X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
        X_v, y_v, test_size=0.2, random_state=42, stratify=y_v
    )
    print("→ Stratified split")
except ValueError:
    X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
        X_v, y_v, test_size=0.2, random_state=42
    )
    print("→ Regular split (could not stratify due to small class counts)")

# ----------------- 4) tensors ON CPU (intentionally) ------------------------------------------------
device = torch.device("cpu")
X_train_tensor_v = torch.tensor(X_train_v, dtype=torch.float32, device=device)
X_test_tensor_v  = torch.tensor(X_test_v,  dtype=torch.float32, device=device)
y_train_tensor_v = torch.tensor(y_train_v, dtype=torch.long,    device=device)

# ----------------- 5) model / opt / loss ------------------------------------------------------------
input_dim_v      = X_train_tensor_v.shape[1]
classifier_video = MLPClassifier(input_dim_v, hidden_dim, num_classes).to(device)
optimizer_v      = optim.Adam(classifier_video.parameters(), lr=lr_mlp)  # CPU → no GPU state
loss_v           = nn.CrossEntropyLoss()

# ----------------- 6) training (CPU) ----------------------------------------------------------------
train_loader_v = DataLoader(
    TensorDataset(X_train_tensor_v, y_train_tensor_v),
    batch_size=min(batch_size_mlp, len(X_train_tensor_v)),  # guards tiny splits
    shuffle=True,
    num_workers=0
)

for epoch in range(1, epochs_mlp + 1):
    classifier_video.train()
    running_loss = 0.0
    for Xb, yb in train_loader_v:
        optimizer_v.zero_grad(set_to_none=True)
        out = classifier_video(Xb)
        l   = loss_v(out, yb)
        l.backward()
        optimizer_v.step()
        running_loss += l.item()
    print(f"[MLP-Video] Epoch {epoch}/{epochs_mlp}, Loss: {running_loss/len(train_loader_v):.4f}")

# ----------------- 7) evaluation --------------------------------------------------------------------
classifier_video.eval()
with torch.no_grad():
    logits_v = classifier_video(X_test_tensor_v)
    y_pred_v = logits_v.argmax(dim=1).cpu().numpy()

# ----------------- 8) metrics -----------------------------------------------------------------------
acc_v                 = accuracy_score(y_test_v, y_pred_v)
pM_v, rM_v, f1M_v, _  = precision_recall_fscore_support(y_test_v, y_pred_v, average="macro",    zero_division=0)
pW_v, rW_v, f1W_v, _  = precision_recall_fscore_support(y_test_v, y_pred_v, average="weighted", zero_division=0)

print(f"\nVideo Test – acc:{acc_v:.4f}  prec^M:{pM_v:.4f}  rec^M:{rM_v:.4f}  f1^M:{f1M_v:.4f}")
print(f"             prec^W:{pW_v:.4f}  rec^W:{rW_v:.4f}  f1^W:{f1W_v:.4f}")

# ----------------- 9) per-class report --------------------------------------------------------------
labels_present_v = unique_labels(y_test_v, y_pred_v)
print(classification_report(
    y_test_v,
    y_pred_v,
    labels        = labels_present_v,
    target_names  = le_mm.inverse_transform(labels_present_v),
    zero_division = 0
))

# free a bit of memory just in case
gc.collect()


In [ ]:
# ****************************************************************************************************
# BLOCK [17]: PLOT CLASSIFICATION REPORT + OVERALL METRICS  (self-contained)
# ****************************************************************************************************
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.utils.multiclass import unique_labels
from IPython.display import display

# ---- 1) choose loader & device ---------------------------------------------------
loader = globals().get("test_loader", None) or globals().get("val_loader", None)
if loader is None:
    raise RuntimeError("Neither 'test_loader' nor 'val_loader' is available.")

if "model" not in globals():
    raise RuntimeError("Model not found in globals().")

model.eval()
device = next(model.parameters()).device

# ---- 2) run inference & collect y_true / y_pred ---------------------------------
y_true_mm, y_pred_mm = [], []

with torch.no_grad():
    for batch in loader:
        # Batch layout in this notebook: 
        #   0: ids [B, L] (Long), 2: aud [B, 13] (Float), 3: vid [B, N] (Float), -1: lbl [B]
        ids = batch[0].to(device)
        aud = batch[2].to(device)
        vid = batch[3].to(device)
        lbl = batch[-1].to(device)

        logits, _ = model(ids, aud, vid)
        y_pred_mm.extend(logits.argmax(1).cpu().numpy().tolist())
        y_true_mm.extend(lbl.cpu().numpy().tolist())

y_true_mm = np.array(y_true_mm)
y_pred_mm = np.array(y_pred_mm)

# ---- 3) find which labels actually appear, and map ids -> strings ----------------
if "le_mm" not in globals():
    raise RuntimeError("LabelEncoder 'le_mm' not found. Make sure the encoder used for training is in scope.")

labels_present = unique_labels(y_true_mm, y_pred_mm)
target_names  = le_mm.inverse_transform(labels_present)

# ---- 4) build classification report (dict -> DataFrame) --------------------------
report_dict = classification_report(
    y_true_mm, y_pred_mm,
    labels       = labels_present,
    target_names = target_names,
    output_dict  = True,
    zero_division= 0
)

df_report = pd.DataFrame(report_dict).transpose().round(4)

# Show a nice table in-notebook
display(df_report)

# Optional: draw a clean table figure (handy for paper screenshots)
fig, ax = plt.subplots(figsize=(9, 3 + 0.3 * len(df_report)))
ax.axis("off")
ax.table(
    cellText = df_report.values,
    colLabels= df_report.columns,
    rowLabels= df_report.index,
    loc      = "center",
)
fig.tight_layout()
plt.show()

# ---- 5) print headline metrics ---------------------------------------------------
acc_overall = accuracy_score(y_true_mm, y_pred_mm)
pM, rM, f1M, _ = precision_recall_fscore_support(
    y_true_mm, y_pred_mm, average="macro", zero_division=0
)
pW, rW, f1W, _ = precision_recall_fscore_support(
    y_true_mm, y_pred_mm, average="weighted", zero_division=0
)

print(f"\nOverall accuracy           : {acc_overall:.4f}")
print(f"Macro-avg  precision/recall/F1 : {pM:.4f} / {rM:.4f} / {f1M:.4f}")
print(f"Weighted-avg precision/recall/F1: {pW:.4f} / {rW:.4f} / {f1W:.4f}")

# ---- 6) stash for later blocks (e.g., [18] sig. test) ----------------------------
# These arrays are the *clean* accuracies per-sample equivalent (0/1), if you need them:
acc_per_sample = (y_true_mm == y_pred_mm).astype(np.float32)
globals()["acc_clean"] = acc_per_sample  # used by [18] if you don't rerun [17-B]

# Keep also the raw y_true/y_pred for any other analysis:
globals()["y_true_mm"] = y_true_mm
globals()["y_pred_mm"] = y_pred_mm
# ****************************************************************************************************


In [ ]:
# ───── Inspect a single batch to see ordering ─────
probe_loader = globals().get("test_loader", val_loader)
batch = next(iter(probe_loader))
print(f"Batch length = {len(batch)}")
for i, item in enumerate(batch):
    try:
        print(f"{i}: shape={tuple(item.shape)}, dtype={item.dtype}")
    except AttributeError:
        print(f"{i}: {type(item)}")


In [ ]:
# ****************************************************************************************************
# BLOCK [17-B] : BUILD acc_clean AND acc_adv ARRAYS (robust unpack, OOM-safe PGD, 1:1 pairing)
# ****************************************************************************************************
import numpy as np, torch, gc
import torchattacks
from tqdm import tqdm

# 0) start with a clean GPU cache
gc.collect(); torch.cuda.empty_cache()

# 1) pick a loader (prefers test_loader if present)
loader = globals().get("test_loader", None) or globals().get("val_loader", None)
assert loader is not None, "[17-B] Could not find test_loader/val_loader."

# 2) attack hyper-params (honor globals if they exist)
eps_attack = globals().get("eps_attack", globals().get("eps", 0.03))
alpha      = globals().get("alpha", 0.0075)
steps_pgd  = globals().get("steps_pgd", 10)

# 3) micro-batch size for the PGD work (independent of DataLoader batch size)
pgd_micro_bs = 2                 # drop to 1 if you still run tight on VRAM

# 4) how to combine audio-PGD and video-PGD into ONE adversarial score per sample
#    "worst" = correct only if BOTH attacks are correct; "mean" = average of two
combine_mode = "worst"

# 5) model to eval/GPU
model.eval().to(device)

# 6) helper: robust batch unpack (handles optional txt_mask / extras)
def unpack_batch(batch):
    """
    Try to identify:
      ids : Long [B, L]
      aud : Float [B, audio_feat_dim]
      vid : Float [B, video_feat_dim]
      lbl : Long  [B]
    """
    ids = aud = vid = lbl = None
    AFD = int(globals().get("audio_feat_dim", 13))
    VFD = int(globals().get("video_feat_dim", 345600))

    for x in batch:
        if not isinstance(x, torch.Tensor):
            continue
        if ids is None and x.dtype == torch.long and x.dim() == 2:
            ids = x; continue
        if aud is None and x.dtype.is_floating_point and x.dim() == 2 and x.shape[-1] == AFD:
            aud = x; continue
        if vid is None and x.dtype.is_floating_point and x.dim() == 2 and x.shape[-1] == VFD:
            vid = x; continue
        if lbl is None and x.dtype == torch.long and x.dim() == 1:
            lbl = x

    # fallbacks
    if ids is None: ids = batch[0]
    if aud is None:
        for x in batch:
            if isinstance(x, torch.Tensor) and x.dtype.is_floating_point and x.dim()==2 and x.shape[-1]==AFD:
                aud = x; break
    if vid is None:
        for x in batch:
            if isinstance(x, torch.Tensor) and x.dtype.is_floating_point and x.dim()==2 and x.shape[-1]==VFD:
                vid = x; break
    if lbl is None and isinstance(batch[-1], torch.Tensor) and batch[-1].dtype == torch.long:
        lbl = batch[-1]

    if any(z is None for z in (ids, aud, vid, lbl)):
        shapes = [tuple(x.shape) for x in batch if isinstance(x, torch.Tensor)]
        raise ValueError(f"[17-B] Could not unpack batch. Saw shapes: {shapes}")
    return ids, aud, vid, lbl

# 7) wrappers: detach non-attacked branches during PGD
class AudioWrapper(torch.nn.Module):
    def __init__(self, base, txt_ids, vid_vec):
        super().__init__()
        self.base, self.txt_ids, self.vid_vec = base, txt_ids, vid_vec
    def forward(self, aud_vec):
        with torch.no_grad():
            t = self.base.text_enc(self.txt_ids)
            v = self.base.video_enc(self.vid_vec)
        a = self.base.audio_enc(aud_vec)
        fused  = self.base.fusion(torch.cat([t, a, v], dim=1))
        return self.base.classifier(fused)

class VideoWrapper(torch.nn.Module):
    def __init__(self, base, txt_ids, aud_vec):
        super().__init__()
        self.base, self.txt_ids, self.aud_vec = base, txt_ids, aud_vec
    def forward(self, vid_vec):
        with torch.no_grad():
            t = self.base.text_enc(self.txt_ids)
            a = self.base.audio_enc(self.aud_vec)
        v = self.base.video_enc(vid_vec)
        fused  = self.base.fusion(torch.cat([t, a, v], dim=1))
        return self.base.classifier(fused)

# 8) main loop (CPU → GPU micro-slices)
acc_clean, acc_adv = [], []

for batch in tqdm(loader):
    ids_cpu, aud_cpu, vid_cpu, lbl_cpu = unpack_batch(batch)
    N = ids_cpu.size(0)

    for s in range(0, N, pgd_micro_bs):
        e = min(s + pgd_micro_bs, N)

        # move only the slice to GPU
        ids_b = ids_cpu[s:e].to(device, non_blocking=True)
        aud_b = aud_cpu[s:e].to(device, non_blocking=True)
        vid_b = vid_cpu[s:e].to(device, non_blocking=True)
        lbl_b = lbl_cpu[s:e].to(device, non_blocking=True)

        # clean inference
        with torch.no_grad():
            log_clean, _ = model(ids_b, aud_b, vid_b)
        acc_clean.extend((log_clean.argmax(1) == lbl_b).cpu().numpy())

        # audio PGD
        audio_model = AudioWrapper(model, ids_b, vid_b).eval().to(device)
        pgd_audio   = torchattacks.PGD(audio_model, eps=eps_attack,
                                       alpha=alpha, steps=steps_pgd, random_start=True)
        adv_aud_b   = pgd_audio(aud_b, lbl_b)

        # video PGD
        video_model = VideoWrapper(model, ids_b, aud_b).eval().to(device)
        pgd_video   = torchattacks.PGD(video_model, eps=eps_attack,
                                       alpha=alpha, steps=steps_pgd, random_start=True)
        adv_vid_b   = pgd_video(vid_b, lbl_b)

        # evaluate both attacks
        with torch.no_grad():
            log_adv_a, _ = model(ids_b, adv_aud_b, vid_b)
            log_adv_v, _ = model(ids_b, aud_b,    adv_vid_b)

        ok_a = (log_adv_a.argmax(1) == lbl_b).cpu().numpy().astype(np.float32)
        ok_v = (log_adv_v.argmax(1) == lbl_b).cpu().numpy().astype(np.float32)
        if combine_mode == "mean":
            ok = (ok_a + ok_v) / 2.0
        else:  # "worst"
            ok = np.minimum(ok_a, ok_v)
        acc_adv.extend(ok)

        # free this slice
        del ids_b, aud_b, vid_b, lbl_b, audio_model, video_model
        del pgd_audio, pgd_video, adv_aud_b, adv_vid_b
        del log_clean, log_adv_a, log_adv_v
        torch.cuda.empty_cache()

# 9) numpy arrays for later blocks
acc_clean = np.asarray(acc_clean, dtype=np.float32).ravel()
acc_adv   = np.asarray(acc_adv,   dtype=np.float32).ravel()
print(f"[17-B] Collected {len(acc_clean)} clean / {len(acc_adv)} adversarial (combine='{combine_mode}', micro_bs={pgd_micro_bs}).")
# ****************************************************************************************************


In [ ]:
# ****************************************************************************************************
# BLOCK [18] : TABLE II — STATISTICAL SIGNIFICANCE (paired t-test clean vs adversarial)
# ****************************************************************************************************
from scipy import stats
import pandas as pd, numpy as np

# ensure proper 1-D arrays
acc_clean = np.asarray(acc_clean, dtype=float).ravel()
acc_adv   = np.asarray(acc_adv,   dtype=float).ravel()

# if lengths mismatch (e.g., legacy acc_adv stored twice/sample), repair once
if acc_adv.size != acc_clean.size:
    print(f"[18] acc_adv len {acc_adv.size} != acc_clean len {acc_clean.size} — repairing worst-case pairs.")
    N = acc_clean.size
    half = acc_adv.size // 2
    if half >= N:
        acc_adv = np.minimum(acc_adv[:N], acc_adv[half:half+N]).astype(float)
    else:
        raise ValueError("[18] Cannot pair acc_adv with acc_clean. Re-run Block [17-B].")

# paired t-test
t_stat, p_val = stats.ttest_rel(acc_clean, acc_adv, nan_policy="omit")

eps_label = globals().get("eps_attack", globals().get("eps", 0.03))

table2 = pd.DataFrame({
    "Setting"         : ["Clean", f"Adversarial (ε={eps_label})"],
    "Mean Acc"        : [acc_clean.mean(), acc_adv.mean()],
    "Std"             : [acc_clean.std(ddof=1), acc_adv.std(ddof=1)],
    "p-Value vs Clean": [np.nan, f"{p_val:.3e}"],
    "N (pairs)"       : [acc_clean.size, acc_adv.size]
})
display(table2)   # ← Table II (Section 4.5)
# ****************************************************************************************************


In [ ]:
# ****************************************************************************************************
# BLOCK [19] : FIGURE 5 — ROBUSTNESS CURVE (Accuracy vs ε on AUDIO)
# ****************************************************************************************************
import numpy as np, torch, gc
import torchattacks
import matplotlib.pyplot as plt
from tqdm import tqdm

gc.collect(); torch.cuda.empty_cache()
model.eval().to(device)

# 1) eps sweep and data loader
eps_list = [0.0, 0.01, 0.02, 0.03, 0.05]
loader   = globals().get("test_loader", None) or globals().get("val_loader", None)
assert loader is not None, "[19] Could not find test_loader/val_loader."

# 2) robust batch unpack (same logic as [17-B])
def unpack_batch(batch):
    ids = aud = vid = lbl = None
    AFD = int(globals().get("audio_feat_dim", 13))
    VFD = int(globals().get("video_feat_dim", 345600))
    for x in batch:
        if not isinstance(x, torch.Tensor): 
            continue
        if ids is None and x.dtype == torch.long and x.dim() == 2:
            ids = x; continue
        if aud is None and x.dtype.is_floating_point and x.dim() == 2 and x.shape[-1] == AFD:
            aud = x; continue
        if vid is None and x.dtype.is_floating_point and x.dim() == 2 and x.shape[-1] == VFD:
            vid = x; continue
        if lbl is None and x.dtype == torch.long and x.dim() == 1:
            lbl = x
    if ids is None: ids = batch[0]
    if aud is None:
        for x in batch:
            if isinstance(x, torch.Tensor) and x.dtype.is_floating_point and x.dim()==2 and x.shape[-1]==AFD:
                aud = x; break
    if vid is None:
        for x in batch:
            if isinstance(x, torch.Tensor) and x.dtype.is_floating_point and x.dim()==2 and x.shape[-1]==VFD:
                vid = x; break
    if lbl is None and isinstance(batch[-1], torch.Tensor) and batch[-1].dtype == torch.long:
        lbl = batch[-1]
    if any(z is None for z in (ids, aud, vid)):
        shapes = [tuple(x.shape) for x in batch if isinstance(x, torch.Tensor)]
        raise ValueError(f"[19] Could not unpack batch. Saw shapes: {shapes}")
    return ids, aud, vid, lbl

# 3) wrapper that uses PRECOMPUTED text/video embeddings (no BERT inside PGD!)
class AudioOnlyPGD(torch.nn.Module):
    def __init__(self, base, t_fixed, v_fixed):
        super().__init__()
        self.base = base
        self.t_fixed = t_fixed.detach()   # [B, Ft]
        self.v_fixed = v_fixed.detach()   # [B, Fv]
    def forward(self, aud_vec):           # [B, A]
        a = self.base.audio_enc(aud_vec)
        fused = self.base.fusion(torch.cat([self.t_fixed, a, self.v_fixed], dim=1))
        return self.base.classifier(fused)

# 4) micro-batch size on GPU (independent of DataLoader batch)
micro_bs = 2  # drop to 1 if memory is still tight

acc_vs_eps = []

for eps in eps_list:
    correct, total = 0, 0
    for batch in tqdm(loader, desc=f"ε={eps:.2f}"):
        ids_cpu, aud_cpu, vid_cpu, lbl_cpu = unpack_batch(batch)
        N = ids_cpu.size(0)

        for s in range(0, N, micro_bs):
            e = min(s + micro_bs, N)

            # slice → GPU
            ids_b = ids_cpu[s:e].to(device, non_blocking=True)
            aud_b = aud_cpu[s:e].to(device, non_blocking=True)
            vid_b = vid_cpu[s:e].to(device, non_blocking=True)
            lbl_b = (lbl_cpu[s:e].to(device, non_blocking=True)
                     if lbl_cpu is not None else None)

            # precompute fixed text/video encodings once (no grad!)
            with torch.no_grad():
                t_fixed = model.text_enc(ids_b)
                v_fixed = model.video_enc(vid_b)

            # build tiny model seen by PGD: only audio path + fusion + head
            tiny = AudioOnlyPGD(model, t_fixed, v_fixed).eval().to(device)

            # PGD on audio only (ensure alpha>0 even for eps=0)
            atk = torchattacks.PGD(
                tiny, eps=eps, alpha=max(eps/4.0, 1e-4), steps=10, random_start=True
            )
            adv_aud_b = atk(aud_b, lbl_b)

            # evaluate attacked audio with original text+video
            with torch.no_grad():
                logits, _ = model(ids_b, adv_aud_b, vid_b)
            correct += (logits.argmax(1) == lbl_b).sum().item()
            total   += lbl_b.size(0)

            # tidy up this slice
            del ids_b, aud_b, vid_b, lbl_b, t_fixed, v_fixed, tiny, atk, adv_aud_b, logits
            torch.cuda.empty_cache()

    acc_vs_eps.append(correct / total)

# 5) plot
plt.plot(eps_list, acc_vs_eps, marker="o")
plt.xlabel("ε")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Attack Strength (audio PGD)")
plt.grid(True); plt.show()
# ****************************************************************************************************


In [ ]:
# ****************************************************************************************************
# BLOCK [20] : Audio & Video Integrated Gradients — CPU-safe, shape-proof, OOM-safe
# ****************************************************************************************************
import gc, math, torch, numpy as np, matplotlib.pyplot as plt
from captum.attr import IntegratedGradients

# 0) housekeeping
gc.collect(); torch.cuda.empty_cache()
model.eval()

# 1) force model to CPU for IG (keeps GPU free)
device_cpu = torch.device("cpu")
model_cpu  = model.to(device_cpu).eval()

# 2) grab ONE validation/test sample (B=1) and move to CPU
batch = next(iter(val_loader if 'val_loader' in globals() and val_loader is not None
                  else test_loader))
ids = batch[0][:1].to(device_cpu)    # [1, L]  (Long)
aud = batch[2][:1].to(device_cpu)    # [1, 13]
vid = batch[3][:1].to(device_cpu)    # [1, 345600] (flattened)
lbl = int(batch[-1][0].item())       # scalar index

# 3) wrapper for IG — replicates text ids for each IG internal micro-step
def f_cpu(a, v):
    B = a.size(0)                                # Captum will call with B = n_steps+1
    ids_exp = ids.expand(B, -1)                  # keep text fixed
    logits, _ = model_cpu(ids_exp, a, v)         # model returns (logits, fused)
    return logits                                # Captum expects just the tensor

ig = IntegratedGradients(f_cpu)

# 4) run IG on CPU (keep internal_batch_size small to cap RAM)
attr_a, attr_v = ig.attribute(
    inputs=(aud, vid),
    target=lbl,
    n_steps=32,
    internal_batch_size=4
)

# 5) ---------- AUDIO attribution (bar plot over 13 features) --------------------
aud_vec = aud[0].cpu().numpy()
attr_a  = attr_a[0].cpu().numpy()                # shape (13,)
plt.figure(figsize=(6,3))
plt.bar(np.arange(len(aud_vec)), aud_vec, label="Audio feature", alpha=0.6)
plt.bar(np.arange(len(attr_a)),  np.abs(attr_a), label="|IG score|", alpha=0.6)
plt.xlabel("Audio feature index"); plt.ylabel("Value / Importance")
plt.title("Audio attribution (bar)"); plt.legend(); plt.tight_layout(); plt.show()

# 6) ---------- VIDEO attribution (1-D line plot over all elements) --------------
# If you want a faster plot, optionally downsample the curve:
v_full = attr_v[0].cpu().numpy().ravel()         # length = video_feat_dim
# optional downsample for speed/readability:
max_points = 5000
if v_full.size > max_points:
    idx = np.linspace(0, v_full.size-1, max_points).astype(int)
    v_plot = v_full[idx]
else:
    v_plot = v_full

plt.figure(figsize=(6,3))
plt.plot(v_plot)
plt.xlabel("Video feature index"); plt.ylabel("IG score")
plt.title("Video attribution (line plot)")
plt.tight_layout(); plt.show()

# 7) (optional) move model back to GPU ONLY if there is room
RETURN_TO_GPU = False   # <- leave False to avoid the OOM you hit
if RETURN_TO_GPU and torch.cuda.is_available():
    # aggressively free CPU/GPU objects first
    del ig, attr_a, attr_v, v_full, v_plot, aud_vec
    gc.collect(); torch.cuda.empty_cache()
    # simple guard: need at least ~2.0 GB free before moving BERT back
    try:
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / (1024**3)
    except Exception:
        free_gb = 0.0
    if free_gb >= 2.0:
        model.to(torch.device("cuda"))
    else:
        print(f"[20] Skipping model.to(cuda): only {free_gb:.2f} GB free.")

# 8) tidy up
gc.collect(); torch.cuda.empty_cache()
# ****************************************************************************************************


In [ ]:
# ****************************************************************************************************
# BLOCK [21]:  TABLE III – SOTA COMPARISON ON IEMOCAP
# ****************************************************************************************************
sota = {
    "RAVEN (2019)"            : 0.724,
    "MulT (2021)"             : 0.784,
    "MAG-BERT (2020)"         : 0.801,
    "TFN (ours, re-run)"      : 0.765,
    "AR-XMER (ours)"          : acc_clean.mean()        # from your eval
}
table3 = pd.DataFrame(list(sota.items()), columns=["Model", "Accuracy"]).sort_values("Accuracy", ascending=False)
display(table3)      # ← copy into Table III (Section 4.8)


In [ ]:
# ****************************************************************************************************
# BLOCK [22] : TABLE IV – FLOPs & LATENCY  (pretty Styler, plus baseline rows)
# ****************************************************************************************************
import time, torch, pandas as pd, importlib, numpy as np

# ─── 1) choose available profiler ───────────────────────────────────────────────
profile_func, prof_src = None, None

if importlib.util.find_spec("thop"):
    from thop import profile                                      # type: ignore
    profile_func = lambda m, inp: profile(m, inp)[0:2]            # GFLOPs, params
    prof_src = "thop"
elif importlib.util.find_spec("torchinfo"):
    from torchinfo import summary                                 # type: ignore
    def profile_func(m, inp):
        macs = summary(m, input_data=inp, verbose=0).total_mult_adds
        return macs * 2 / 1e9, m.num_parameters()                 # GFLOPs, params
    prof_src = "torchinfo"
else:
    raise RuntimeError(
        "Neither 'thop' nor 'torchinfo' found in environment. "
        "Load one before running this block."
    )

# ─── 2) dummy inputs & model on CPU to avoid GPU OOM ────────────────────────────
device_cpu = torch.device("cpu")
model_cpu  = model.to(device_cpu).eval()

dummy_ids  = torch.randint(0, tokenizer.vocab_size, (1, 128), device=device_cpu)
dummy_aud  = torch.zeros(1, audio_feat_dim, device=device_cpu)
dummy_vid  = torch.zeros(1, video_feat_dim, device=device_cpu)

# ─── 3) FLOPs (G) via profiler ─────────────────────────────────────────────────
flops_G, _ = profile_func(model_cpu, (dummy_ids, dummy_aud, dummy_vid))
flops_G    = round(flops_G, 3)

# ─── 4) latency (ms) – average over n_rep CPU passes ───────────────────────────
n_rep, t0 = 50, time.time()
with torch.no_grad():
    for _ in range(n_rep):
        model_cpu(dummy_ids, dummy_aud, dummy_vid)
lat_ms = round((time.time() - t0) / n_rep * 1000, 3)

# ─── 5) build our model’s row ───────────────────────────────────────────────────
rows = [{
    "Model"       : "AR-XMER (ours)",
    "FLOPs (G)"   : flops_G,
    "Latency (ms)": lat_ms,
    "Profiler"    : prof_src
}]

# ─── 6) add baseline rows **manually here**  ───────────────────────────────────
#    → replace the FLOPs / latency with real numbers from papers or your own runs
rows += [
    {"Model": "RAVEN (2019)",    "FLOPs (G)": 0.724,      "Latency (ms)": np.nan, "Profiler": "paper"},
    {"Model": "BERT-MER (2020)", "FLOPs (G)": 1.005,      "Latency (ms)": np.nan, "Profiler": "paper"},
    # add / edit more baselines as needed…
]

# ─── 7) render nice table with Pandas Styler ────────────────────────────────────
table4 = pd.DataFrame(rows)

display(
    table4
        .style
        .format({"FLOPs (G)": "{:.3f}", "Latency (ms)": "{:.1f}"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
        .set_properties(**{"text-align": "center"})
        .hide(axis="index")                        # hide integer index column
)
# ****************************************************************************************************


In [ ]:
assert "model" in globals(), "Block 23 needs `model` (load/train it first in this kernel)."
assert ("test_loader" in globals()) or ("val_loader" in globals()), "Block 23 needs `test_loader` or `val_loader`."

In [ ]:
# ****************************************************************************************************
# BLOCK [23] : ERI RESULTS (mean/std/N) ON CLEAN vs AUDIO-PGD PAIRS  (fast, no retraining)
# ****************************************************************************************************
import numpy as np, torch, gc
from captum.attr import IntegratedGradients

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------
# Settings (match your paper)
# ----------------------------
loader = globals().get("test_loader", None) or globals().get("val_loader", None)
assert loader is not None, "[21] Could not find test_loader/val_loader."

MAX_PAIRS = int(globals().get("ERI_MAX_PAIRS", 31))   # match Table IV (31 pairs)
EPS_ERI   = float(globals().get("ERI_EPS", 0.01))     # Table IV uses ε = 0.01
ALPHA_ERI = float(globals().get("ERI_ALPHA", 0.002))  # step size (safe default for ε=0.01)
STEPS_ERI = int(globals().get("ERI_STEPS", 10))       # PGD steps
TOPK_FRAC = float(globals().get("ERI_TOPK_FRAC", 0.10))  # top-k fraction for binarising saliency

model.eval().to(device)

# ----------------------------
# Helper: unpack a batch robustly
# ----------------------------
def unpack_batch(batch):
    ids = aud = vid = lbl = None
    AFD = int(globals().get("audio_feat_dim", 13))
    VFD = int(globals().get("video_feat_dim", 345600))

    for x in batch:
        if not isinstance(x, torch.Tensor):
            continue
        if ids is None and x.dtype == torch.long and x.dim() == 2:
            ids = x
            continue
        if aud is None and x.dtype.is_floating_point and x.dim() == 2 and x.shape[-1] == AFD:
            aud = x
            continue
        if vid is None and x.dtype.is_floating_point and x.dim() == 2 and x.shape[-1] == VFD:
            vid = x
            continue
        if lbl is None and x.dtype == torch.long and x.dim() == 1:
            lbl = x
            continue

    assert ids is not None and aud is not None and vid is not None and lbl is not None, \
        "[21] Could not unpack (ids, aud, vid, lbl)."

    return ids, aud, vid, lbl

# ----------------------------
# Helper: PGD on AUDIO only (L_inf)
# ----------------------------
def pgd_audio(ids, aud, vid, y_target, eps=0.01, alpha=0.002, steps=10):
    """
    Untargeted PGD that maximizes CE loss wrt y_target (here: predicted clean class).
    """
    aud0 = aud.detach()
    aud_adv = aud0.clone().detach()

    # small random start within the epsilon ball
    aud_adv = aud_adv + torch.empty_like(aud_adv).uniform_(-eps, eps)
    aud_adv = torch.max(torch.min(aud_adv, aud0 + eps), aud0 - eps).detach()

    for _ in range(steps):
        aud_adv.requires_grad_(True)
        model.zero_grad(set_to_none=True)

        logits, _ = model(ids, aud_adv, vid)
        loss = torch.nn.functional.cross_entropy(logits, y_target)

        loss.backward()
        g = aud_adv.grad.detach().sign()

        aud_adv = aud_adv.detach() + alpha * g
        aud_adv = torch.max(torch.min(aud_adv, aud0 + eps), aud0 - eps).detach()

    return aud_adv.detach()

# ----------------------------
# Helper: ERI from IG on fused-input vector (text_emb || audio_emb || video_emb)
# ----------------------------
def topk_mask(x_1d, frac=0.10):
    x = np.abs(x_1d).astype(np.float32)
    if x.size == 0:
        return np.zeros_like(x, dtype=np.uint8)
    k = max(1, int(np.ceil(frac * x.size)))
    thr = np.partition(x, -k)[-k]
    return (x >= thr).astype(np.uint8)

def iou(mask_a, mask_b):
    inter = (mask_a & mask_b).sum()
    union = (mask_a | mask_b).sum()
    return float(inter) / float(union + 1e-8)

def eri_from_attr(attr_clean_concat, attr_adv_concat, t_dim, a_dim, v_dim, topk_frac=0.10):
    ac = attr_clean_concat.reshape(-1)
    aa = attr_adv_concat.reshape(-1)

    ac_t, ac_a, ac_v = ac[:t_dim], ac[t_dim:t_dim+a_dim], ac[t_dim+a_dim:t_dim+a_dim+v_dim]
    aa_t, aa_a, aa_v = aa[:t_dim], aa[t_dim:t_dim+a_dim], aa[t_dim+a_dim:t_dim+a_dim+v_dim]

    m_ct = topk_mask(ac_t, topk_frac); m_at = topk_mask(aa_t, topk_frac)
    m_ca = topk_mask(ac_a, topk_frac); m_aa = topk_mask(aa_a, topk_frac)
    m_cv = topk_mask(ac_v, topk_frac); m_av = topk_mask(aa_v, topk_frac)

    return (iou(m_ct, m_at) + iou(m_ca, m_aa) + iou(m_cv, m_av)) / 3.0

# ----------------------------
# IG wrapper: attribution wrt the concatenated embedding input
# ----------------------------
def logits_from_concat(concat_emb):
    fused = model.fusion(concat_emb)
    return model.classifier(fused)

ig = IntegratedGradients(logits_from_concat)

eri_vals = []
n_done = 0

with torch.no_grad():
    # infer dims once
    tmp_batch = next(iter(loader))
    ids0, aud0, vid0, lbl0 = unpack_batch(tmp_batch)
    ids0, aud0, vid0, lbl0 = ids0.to(device), aud0.to(device), vid0.to(device), lbl0.to(device)
    t0 = model.text_enc(ids0[:1])
    a0 = model.audio_enc(aud0[:1])
    v0 = model.video_enc(vid0[:1])
    T_DIM, A_DIM, V_DIM = t0.shape[1], a0.shape[1], v0.shape[1]

for batch in loader:
    ids, aud, vid, lbl = unpack_batch(batch)
    ids, aud, vid, lbl = ids.to(device), aud.to(device), vid.to(device), lbl.to(device)

    B = ids.size(0)
    for i in range(B):
        if n_done >= MAX_PAIRS:
            break

        ids_i = ids[i:i+1]
        aud_i = aud[i:i+1]
        vid_i = vid[i:i+1]

        # clean prediction (we explain the predicted class, consistent with your Section III-F definition)
        model.eval()
        with torch.no_grad():
            logits_clean, _ = model(ids_i, aud_i, vid_i)
            y_hat = logits_clean.argmax(dim=1)

            t = model.text_enc(ids_i)
            a = model.audio_enc(aud_i)
            v = model.video_enc(vid_i)
            concat_clean = torch.cat([t, a, v], dim=1)

        # adversarial audio (PGD)
        aud_adv = pgd_audio(ids_i, aud_i, vid_i, y_hat, eps=EPS_ERI, alpha=ALPHA_ERI, steps=STEPS_ERI)

        with torch.no_grad():
            a_adv = model.audio_enc(aud_adv)
            concat_adv = torch.cat([t, a_adv, v], dim=1)

        # IG on concat vectors (keep internal_batch_size small to avoid RAM spikes)
        attr_clean = ig.attribute(
            inputs=concat_clean,
            baselines=torch.zeros_like(concat_clean),
            target=int(y_hat.item()),
            n_steps=16,
            internal_batch_size=4
        ).detach().cpu().numpy()

        attr_adv = ig.attribute(
            inputs=concat_adv,
            baselines=torch.zeros_like(concat_adv),
            target=int(y_hat.item()),
            n_steps=16,
            internal_batch_size=4
        ).detach().cpu().numpy()

        eri_i = eri_from_attr(attr_clean, attr_adv, T_DIM, A_DIM, V_DIM, topk_frac=TOPK_FRAC)
        eri_vals.append(eri_i)

        n_done += 1

    if n_done >= MAX_PAIRS:
        break

eri_vals = np.asarray(eri_vals, dtype=np.float32)
eri_mean = float(eri_vals.mean()) if eri_vals.size else float("nan")
eri_std  = float(eri_vals.std(ddof=1)) if eri_vals.size > 1 else float("nan")

print(f"[ERI] eps={EPS_ERI} alpha={ALPHA_ERI} steps={STEPS_ERI} topk_frac={TOPK_FRAC}")
print(f"[ERI] N={eri_vals.size}  mean={eri_mean:.4f}  std={eri_std:.4f}")